# 16 · Top 100 LeetCode questions — part 2: sliding window, stacks and binary search

After this notebook you can solve 20 of the most-asked **easy and medium** questions about windows, piles and
halving — and, more importantly, recognise *which* of these three tricks a brand-new question wants. A **sliding
window** looks at a stretch of boxes that grows on the right and shrinks on the left. A **stack** is a pile of
plates where you only touch the top one. **Binary search** is the "guess the number, higher or lower?" game: every
guess throws half of the choices away. Every answer is plain Python 3, checked with `assert`s, and comes with an
animation you can step through box by box.

**Time:** ~60 min · **Runs:** offline on CPU in seconds · **Needs:** [03 · Hashing and sliding window](03_hashing_and_sliding_window.ipynb), [04 · Stacks and queues](04_stacks_and_queues.ipynb), [06 · Binary search](06_binary_search.ipynb)

## How to use this notebook

1. **Read** the 🧸 story and the question.
2. **Cover the code and try it yourself** for 10 minutes — even just the idea in words counts.
3. **Run** the answer, then press **▶ play** on the animation and watch every step. Use **◀ back** and **next ▶**
   when a step surprises you.
4. **Tomorrow**, solve it again from a blank cell. That is what makes it stick.

The colours always mean the same thing: yellow = looking at it now, green = yes / the answer, red = no, grey =
finished (or thrown away), blue = inside the window, orange = moving, purple = just added.

## The 20 in this notebook

| # | Problem | LeetCode | Level | Trick |
|---|---|---|---|---|
| 21 | Best Time to Buy and Sell Stock | [121](https://leetcode.com/problems/best-time-to-buy-and-sell-stock/) | 🟢 Easy | remember the cheapest day so far |
| 22 | Best Time to Buy and Sell Stock II | [122](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-ii/) | 🟡 Medium | add up every climb |
| 23 | Longest Substring Without Repeating Characters | [3](https://leetcode.com/problems/longest-substring-without-repeating-characters/) | 🟡 Medium | sliding window + last-seen index |
| 24 | Longest Repeating Character Replacement | [424](https://leetcode.com/problems/longest-repeating-character-replacement/) | 🟡 Medium | sliding window: size − most common ≤ k |
| 25 | Permutation in String | [567](https://leetcode.com/problems/permutation-in-string/) | 🟡 Medium | fixed window + letter counts |
| 26 | Minimum Size Subarray Sum | [209](https://leetcode.com/problems/minimum-size-subarray-sum/) | 🟡 Medium | grow right, shrink left while big enough |
| 27 | Find All Anagrams in a String | [438](https://leetcode.com/problems/find-all-anagrams-in-a-string/) | 🟡 Medium | fixed window + letter counts |
| 28 | Valid Parentheses | [20](https://leetcode.com/problems/valid-parentheses/) | 🟢 Easy | stack of openers |
| 29 | Min Stack | [155](https://leetcode.com/problems/min-stack/) | 🟡 Medium | a second stack of "smallest so far" |
| 30 | Evaluate Reverse Polish Notation | [150](https://leetcode.com/problems/evaluate-reverse-polish-notation/) | 🟡 Medium | stack of numbers |
| 31 | Daily Temperatures | [739](https://leetcode.com/problems/daily-temperatures/) | 🟡 Medium | monotonic stack |
| 32 | Decode String | [394](https://leetcode.com/problems/decode-string/) | 🟡 Medium | stack of saved work |
| 33 | Implement Queue using Stacks | [232](https://leetcode.com/problems/implement-queue-using-stacks/) | 🟢 Easy | two stacks: in and out |
| 34 | Binary Search | [704](https://leetcode.com/problems/binary-search/) | 🟢 Easy | binary search |
| 35 | Search Insert Position | [35](https://leetcode.com/problems/search-insert-position/) | 🟢 Easy | binary search, return L |
| 36 | Search a 2D Matrix | [74](https://leetcode.com/problems/search-a-2d-matrix/) | 🟡 Medium | binary search on one long row |
| 37 | Find Minimum in Rotated Sorted Array | [153](https://leetcode.com/problems/find-minimum-in-rotated-sorted-array/) | 🟡 Medium | binary search: compare with the right end |
| 38 | Search in Rotated Sorted Array | [33](https://leetcode.com/problems/search-in-rotated-sorted-array/) | 🟡 Medium | binary search: one half is always sorted |
| 39 | Find First and Last Position of Element in Sorted Array | [34](https://leetcode.com/problems/find-first-and-last-position-of-element-in-sorted-array/) | 🟡 Medium | binary search twice, keep going on a hit |
| 40 | Koko Eating Bananas | [875](https://leetcode.com/problems/koko-eating-bananas/) | 🟡 Medium | binary search on the answer |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), Path.cwd() / "03_dsa", *Path.cwd().parents]:
    if (_p / "dsa_anim.py").exists():
        sys.path.insert(0, str(_p)); break
from dsa_anim import Movie, arr, bars, hmap, stack, queue, chain, chain_of, tree, grid, graph, note, say
import random
from collections import Counter, deque


def grey_before(i):
    """Colour boxes 0..i-1 grey: the ones we already walked past."""
    return {k: "done" for k in range(i)}


def grey_outside(n, left, right):
    """Colour every box outside left..right grey: the part binary search has thrown away."""
    return {k: "done" for k in range(n) if k < left or k > right}

## The tricks in this notebook, in pictures

Three small movies, one per trick. They are not tied to any one problem — they show the *shape* of the idea, so
you can spot it again later.

- **A window slides along.** Instead of adding up every stretch from scratch, move a window one box at a time:
  one box walks in on the right, one walks out on the left.
- **A stack is a pile of plates.** You can only put a plate on top or take the top plate off. The last one in is
  the first one out.
- **Higher or lower?** To find a number in a sorted row, look at the middle. Too small? Everything to its left is
  too small too — throw that half away. Each guess halves the row, so 1,000 cards need only about 10 guesses.

In [ ]:
# 1. A window slides along: the best total of 3 boxes that sit next to each other.
nums, k = [2, 1, 5, 1, 3, 2], 3
m = Movie("A window slides along", "Best total of 3 boxes in a row")
total = best = sum(nums[:k])
m.step(f"Put a window over the first {k} boxes. Their total is `{total}`.", arr(nums, win=(0, k - 1)), note(total=total, best=best))
for right in range(k, len(nums)):
    left = right - k + 1
    total += nums[right] - nums[left - 1]          # one walks in, one walks out
    best = max(best, total)
    m.step(f"Slide one box: `{nums[right]}` walks **in**, `{nums[left - 1]}` walks **out**. New total `{total}` — no need to add all {k} again.",
           arr(nums, win=(left, right), hi={left - 1: "done", right: "new"}), note(total=total, best=best))
m.step(f"The window reached the end. The best total was **{best}**.", arr(nums), note(best=best))
assert best == max(sum(nums[i:i + k]) for i in range(len(nums) - k + 1)) == 9
m.show()

# 2. A stack is a pile of plates: only the top plate can move.
m = Movie("A stack is a pile of plates", "Last one in is the first one out")
pile = []
m.step("An empty pile. We may only touch the **top**.", stack(pile, "pile"))
for action in ["push A", "push B", "push C", "pop", "push D", "pop", "pop"]:
    if action == "pop":
        top = pile[-1]
        m.step(f"**Pop**: take off the top plate `{top}` — the newest one, not the oldest.", stack(pile, "pile", hi={len(pile) - 1: "swap"}))
        pile.pop()
    else:
        pile.append(action.split()[1])
        m.step(f"**Push** `{pile[-1]}`: it goes on top of the others.", stack(pile, "pile", hi={len(pile) - 1: "new"}))
m.step(f"Only `{pile[0]}` is left: it went in first, so it comes out last.", stack(pile, "pile", hi={0: "good"}))
assert pile == ["A"]
m.show()

# 3. Higher or lower? Find the secret number in a sorted row by halving.
cards, secret = list(range(1, 16)), 11
m = Movie("Higher or lower?", f"Find {secret} among 15 sorted cards")
left, right, guesses = 0, len(cards) - 1, 0
m.step(f"The secret is somewhere between `{cards[left]}` and `{cards[right]}`. Guess the **middle**.",
       arr(cards, ptr={"L": left, "R": right}, index=False))
while left <= right:
    mid = (left + right) // 2
    guesses += 1
    grey = grey_outside(len(cards), left, right)
    if cards[mid] == secret:
        m.step(f"Guess {guesses}: `{cards[mid]}`. Got it! 🎉 Only {guesses} guesses for 15 cards.",
               arr(cards, hi={**grey, mid: "good"}, ptr={"L": left, "M": mid, "R": right}, index=False))
        break
    if cards[mid] < secret:
        m.step(f"Guess {guesses}: `{cards[mid]}`. **Higher!** So `{cards[mid]}` and everything left of it is out.",
               arr(cards, hi={**grey, **{k: "bad" for k in range(left, mid + 1)}}, ptr={"L": left, "M": mid, "R": right}, index=False))
        left = mid + 1
    else:
        m.step(f"Guess {guesses}: `{cards[mid]}`. **Lower!** So `{cards[mid]}` and everything right of it is out.",
               arr(cards, hi={**grey, **{k: "bad" for k in range(mid, right + 1)}}, ptr={"L": left, "M": mid, "R": right}, index=False))
        right = mid - 1
assert cards[mid] == secret and guesses <= 4
m.show()

---

## 21 · Best Time to Buy and Sell Stock · [LeetCode 121](https://leetcode.com/problems/best-time-to-buy-and-sell-stock/) · 🟢 Easy

**🧸 Like you're five:** You can buy one toy car on one day and sell it on a later day. You look at each day's price and keep
        in your head the **cheapest day so far**. Each new day you ask: "if I had bought on that cheap day, how much
        would I win by selling today?"

**The question:** `prices[i]` is the price on day `i`. Buy on one day and sell on a **later** day. Return the biggest profit, or `0` if no trade makes money.

| Input | Output |
|---|---|
| `prices = [7, 1, 5, 3, 6, 4]` | `5` |
| `prices = [7, 6, 4, 3, 1]` | `0` |
| `prices = [5]` | `0` |

**🔎 How to spot it:** "buy **before** you sell", "the biggest difference `later − earlier`" → walk once and remember the
        smallest value seen so far. It is a sliding window whose left end is always the cheapest day.

**💡 The trick (cheapest so far):** Keep two numbers: `cheapest` (the lowest price so far) and `best` (the best profit so far). For each
        price, first update `cheapest`, then try selling today: `best = max(best, price − cheapest)`. The trap: you
        cannot just take `max(prices) − min(prices)` — the maximum might come **before** the minimum
        (`[2, 4, 1]` → the answer is 2, not 3).

**⏱ Cost:** **O(n)** time (one walk) · **O(1)** space (two numbers). Brute force: try every buy day with every later sell day — O(n²).

In [ ]:
def max_profit(prices):
    cheapest = float("inf")               # the lowest price seen so far
    best = 0                              # the best profit so far (0 = don't trade)
    for p in prices:
        cheapest = min(cheapest, p)       # could we have bought cheaper?
        best = max(best, p - cheapest)    # sell today, having bought on the cheapest earlier day
    return best

assert max_profit([7, 1, 5, 3, 6, 4]) == 5
assert max_profit([7, 6, 4, 3, 1]) == 0   # prices only fall: don't trade
assert max_profit([5]) == 0               # one day: nothing to sell
assert max_profit([2, 4, 1]) == 2         # the trap: the minimum comes after the maximum

rng = random.Random(0)
for _ in range(300):
    ps = [rng.randint(0, 20) for _ in range(rng.randint(1, 9))]
    brute = max([0] + [ps[j] - ps[i] for i in range(len(ps)) for j in range(i + 1, len(ps))])
    assert max_profit(ps) == brute
print(max_profit([7, 1, 5, 3, 6, 4]))

In [ ]:
def max_profit_movie(prices):
    m = Movie("Best Time to Buy and Sell Stock", "Buy on one day, sell on a later day. Biggest profit?")
    m.step("Each bar is one day's price. Walk left to right and remember the **cheapest** day so far (blue).",
           bars(prices), note(best=0))
    buy, best, deal = None, 0, None
    for i, p in enumerate(prices):
        past = grey_before(i)
        if buy is None or p < prices[buy]:
            buy = i
            m.step(f"Day {i} costs `{p}`: the cheapest day so far, so it becomes our **buy** day.",
                   bars(prices, hi={**past, i: "win"}, ptr={"buy": i}), note(cheapest=p, best=best))
            continue
        gain = p - prices[buy]
        if gain > best:
            best, deal = gain, (buy, i)
            cap = f"Sell on day {i}? `{p} - {prices[buy]} = {gain}`. Better than before, so **best = {gain}** 🎉"
        else:
            cap = f"Sell on day {i}? `{p} - {prices[buy]} = {gain}`. Not more than `{best}`, so keep walking."
        m.step(cap, bars(prices, hi={**past, buy: "win", i: "good" if deal == (buy, i) else "look"},
                         ptr={"buy": buy, "sell": i}), note(cheapest=prices[buy], best=best))
    if deal:
        b, s = deal
        m.step(f"Answer: **{best}**. Buy at `{prices[b]}` on day {b}, sell at `{prices[s]}` on day {s}.",
               bars(prices, hi={b: "good", s: "good"}, ptr={"buy": b, "sell": s}), note(best=best))
    else:
        m.step("Prices only went down, so the best plan is not to trade: **0**.", bars(prices), note(best=0))
    return best, m

prices = [7, 2, 5, 1, 4, 8, 3]
answer, m = max_profit_movie(prices)
assert answer == max_profit(prices) == 7
m.show()

---

## 22 · Best Time to Buy and Sell Stock II · [LeetCode 122](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-ii/) · 🟡 Medium

**🧸 Like you're five:** Now you may buy and sell as often as you like (but hold only one toy at a time). Whenever tomorrow's
        price is higher than today's, buy today and sell tomorrow. Every little hill you climb puts coins in your
        pocket.

**The question:** `prices[i]` is the price on day `i`. You may hold at most one share, but may buy and sell on as many days as you like (even sell and buy again on the same day). Return the maximum total profit.

| Input | Output |
|---|---|
| `prices = [7, 1, 5, 3, 6, 4]` | `7` |
| `prices = [1, 2, 3, 4, 5]` | `4` |
| `prices = [7, 6, 4, 3, 1]` | `0` |

**🔎 How to spot it:** "as many transactions as you like" with no fee and no cooldown → you can grab **every** rise. (With a
        fee, a cooldown or "at most k trades", it becomes dynamic programming instead.)

**💡 The trick (add up every climb):** Walk through neighbouring days. If `prices[i] > prices[i − 1]`, add the difference. A long climb from
        1 to 6 is the same money as the small steps 1→4→6 added together, so you never need to find the peaks. People
        overthink this one: no need to track buy and sell days at all.

**⏱ Cost:** **O(n)** time · **O(1)** space. Brute force: try every buy/sell/wait choice on every day — O(2ⁿ).

In [ ]:
def max_profit_ii(prices):
    profit = 0
    for i in range(1, len(prices)):
        if prices[i] > prices[i - 1]:              # the price went up since yesterday
            profit += prices[i] - prices[i - 1]    # buy yesterday, sell today
    return profit

assert max_profit_ii([7, 1, 5, 3, 6, 4]) == 7    # (5 - 1) + (6 - 3)
assert max_profit_ii([1, 2, 3, 4, 5]) == 4       # one long climb = the sum of its small steps
assert max_profit_ii([7, 6, 4, 3, 1]) == 0
assert max_profit_ii([1]) == 0

def brute(prices):                                # try every choice on every day
    def go(i, holding):
        if i == len(prices):
            return 0
        wait = go(i + 1, holding)
        act = go(i + 1, False) + prices[i] if holding else go(i + 1, True) - prices[i]
        return max(wait, act)
    return go(0, False)

rng = random.Random(0)
for _ in range(300):
    ps = [rng.randint(0, 9) for _ in range(rng.randint(1, 8))]
    assert max_profit_ii(ps) == brute(ps)
print(max_profit_ii([7, 1, 5, 3, 6, 4]))

In [ ]:
def max_profit_ii_movie(prices):
    m = Movie("Best Time to Buy and Sell Stock II", "Trade as often as you like: collect every climb")
    profit, ups = 0, []
    m.step("Compare each day with the day before. Whenever the price **goes up**, we pocket the difference.",
           bars(prices), note(profit=0))
    for i in range(1, len(prices)):
        past = grey_before(i - 1)
        a, b = prices[i - 1], prices[i]
        if b > a:
            profit += b - a
            ups += [i - 1, i]
            m.step(f"Day {i - 1} → day {i}: `{a}` → `{b}` goes **up**. Buy, then sell: pocket `{b - a}`.",
                   bars(prices, hi={**past, i - 1: "good", i: "good"}, ptr={"buy": i - 1, "sell": i}), note(profit=profit))
        else:
            m.step(f"Day {i - 1} → day {i}: `{a}` → `{b}` goes **down**. Do nothing.",
                   bars(prices, hi={**past, i - 1: "bad", i: "bad"}), note(profit=profit))
    m.step(f"Answer: **{profit}** — the sum of every climb (green). The long climb `1 → 4 → 6` paid the same as one trade `1 → 6`.",
           bars(prices, hi={k: "good" for k in ups}), note(profit=profit))
    return profit, m

prices = [3, 1, 4, 6, 2, 5, 7]
answer, m = max_profit_ii_movie(prices)
assert answer == max_profit_ii(prices) == 10
m.show()

---

## 23 · Longest Substring Without Repeating Characters · [LeetCode 3](https://leetcode.com/problems/longest-substring-without-repeating-characters/) · 🟡 Medium

**🧸 Like you're five:** You are threading letter beads onto a string, and no letter may appear twice. When a bead would repeat,
        you slide beads off the left end until the old copy is gone. What is the longest string you ever hold?

**The question:** Given a string `s`, return the length of the longest **substring** (a stretch of letters next to each other) that has no repeated character.

| Input | Output |
|---|---|
| `s = "abcabcbb"` | `3` |
| `s = "pwwkew"` | `3` |
| `s = ""` | `0` |

**🔎 How to spot it:** "longest **substring / subarray** such that …" and the rule can be fixed by dropping letters from the
        left → a **sliding window** that grows on the right and shrinks on the left.

**💡 The trick (sliding window):** Keep `last[ch]`, the last index where you saw each letter, and the window's left end `left`. For each
        new letter at `right`: if it was seen **inside** the window (`last[ch] >= left`), jump `left` to just after
        that old copy. Then record `last[ch] = right` and measure `right − left + 1`. The trap: an old copy that is
        already **outside** the window must not pull `left` backwards (`"abba"` → 2, not 3).

**⏱ Cost:** **O(n)** time (each letter enters once, `left` only moves forward) · **O(k)** space for `last` (k = size of the alphabet). Brute force: check every substring — O(n³), or O(n²) with a set.

In [ ]:
def length_of_longest_substring(s):
    last = {}                      # letter -> the last index where we saw it
    left = best = 0                # the window is s[left..right]
    for right, ch in enumerate(s):
        if ch in last and last[ch] >= left:    # a repeat INSIDE the window
            left = last[ch] + 1                # jump just past the old copy
        last[ch] = right
        best = max(best, right - left + 1)
    return best

assert length_of_longest_substring("abcabcbb") == 3
assert length_of_longest_substring("bbbbb") == 1
assert length_of_longest_substring("pwwkew") == 3     # "wke" (a substring, not "pwke")
assert length_of_longest_substring("") == 0
assert length_of_longest_substring(" ") == 1          # a space is a character too
assert length_of_longest_substring("abba") == 2       # the old "a" is outside the window: don't move back

rng = random.Random(0)
for _ in range(300):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(0, 10)))
    brute = max([0] + [j - i for i in range(len(s)) for j in range(i + 1, len(s) + 1) if len(set(s[i:j])) == j - i])
    assert length_of_longest_substring(s) == brute
print(length_of_longest_substring("abcabcbb"))

In [ ]:
def longest_substring_movie(s):
    m = Movie("Longest Substring Without Repeating Characters", f"Longest stretch of {s!r} with no letter twice")
    chars, last, left, best, best_at = list(s), {}, 0, 0, (0, -1)
    m.step("The blue window holds letters with **no repeats**. `last` remembers where we last saw each letter.",
           arr(chars), hmap(last, "last seen (letter → index)"), note(best=0))
    for right, ch in enumerate(s):
        if ch in last and last[ch] >= left:
            old = last[ch]
            m.step(f"`{ch}` is already in the window, at index {old}. A repeat! Move L to {old + 1}, just past the old `{ch}`.",
                   arr(chars, win=(left, right), hi={**grey_before(left), old: "bad", right: "bad"}, ptr={"L": left, "R": right}),
                   hmap(last, "last seen (letter → index)", hi={ch: "bad"}), note(best=best))
            left = old + 1
        elif ch in last:
            m.step(f"`{ch}` was seen at index {last[ch]}, but that is **outside** the window (L = {left}). Ignore it — never move L back!",
                   arr(chars, win=(left, right), hi={**grey_before(left), right: "look"}, ptr={"L": left, "R": right}),
                   hmap(last, "last seen (letter → index)", hi={ch: "look"}), note(best=best))
        last[ch] = right
        size = right - left + 1
        if size > best:
            best, best_at = size, (left, right)
        m.step(f"Window `{s[left:right + 1]}` has no repeats: length {size}." + (" A new best!" if best_at == (left, right) else ""),
               arr(chars, win=(left, right), hi={**grey_before(left), right: "new"}, ptr={"L": left, "R": right}),
               hmap(last, "last seen (letter → index)", hi={ch: "new"}), note(best=best))
    a, b = best_at
    m.step(f"Answer: **{best}**, the stretch `{s[a:b + 1]}`.",
           arr(chars, hi={k: "good" for k in range(a, b + 1)}), note(best=best))
    return best, m

s = "tmmzuxt"
answer, m = longest_substring_movie(s)
assert answer == length_of_longest_substring(s) == 5
m.show()

---

## 24 · Longest Repeating Character Replacement · [LeetCode 424](https://leetcode.com/problems/longest-repeating-character-replacement/) · 🟡 Medium

**🧸 Like you're five:** You have a row of red and blue beads and a pot of paint that can recolour **k** beads. What is the
        longest stretch you can make all one colour? Inside any stretch, keep the most common colour and paint the
        rest — as long as "the rest" is at most k.

**The question:** `s` has only uppercase English letters. You may change at most `k` characters to any other uppercase letter. Return the length of the longest substring that can be made of one repeated letter.

| Input | Output |
|---|---|
| `s = "ABAB", k = 2` | `4` |
| `s = "AABABBA", k = 1` | `4` |
| `s = "A", k = 0` | `1` |

**🔎 How to spot it:** "longest substring … **at most k** changes / flips / deletions" → a sliding window with a rule you can
        check from counts. Grow right; when the rule breaks, shrink from the left.

**💡 The trick (sliding window):** A window is fine when `size − (count of its most common letter) ≤ k`: those other letters are the ones
        we would repaint. Add `s[right]` to the counts; while the window is not fine, drop `s[left]` and move `left`
        on. The answer is the biggest fine window. (A famous speed-up keeps the best-ever `most` and never lowers it;
        recomputing `max` over 26 counts is simpler and still O(n).) The trap: keeping the most common letter
        **fixed** — it changes as the window moves.

**⏱ Cost:** **O(26·n) = O(n)** time · **O(26)** space for the counts. Brute force: check every substring — O(n²) or worse.

In [ ]:
def character_replacement(s, k):
    count = {}                      # letter -> how many times it is in the window
    left = best = 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        # letters we would repaint = window size - the most common letter's count
        while (right - left + 1) - max(count.values()) > k:
            count[s[left]] -= 1     # too many to repaint: drop the leftmost letter
            left += 1
        best = max(best, right - left + 1)
    return best

assert character_replacement("ABAB", 2) == 4
assert character_replacement("AABABBA", 1) == 4
assert character_replacement("A", 0) == 1
assert character_replacement("AAAA", 0) == 4
assert character_replacement("ABCDE", 1) == 2
assert character_replacement("ABBB", 5) == 4     # k bigger than the string

rng = random.Random(0)
for _ in range(300):
    letters = rng.choice(["AB", "ABC"])
    s, k = "".join(rng.choice(letters) for _ in range(rng.randint(1, 9))), rng.randint(0, 3)
    brute = max(j - i for i in range(len(s)) for j in range(i + 1, len(s) + 1)
                if (j - i) - max(Counter(s[i:j]).values()) <= k)
    assert character_replacement(s, k) == brute
print(character_replacement("AABABBA", 1))

In [ ]:
def character_replacement_movie(s, k):
    m = Movie("Longest Repeating Character Replacement", f"Repaint at most {k} letter: longest one-letter stretch?")
    chars, count, left, best, best_at = list(s), {}, 0, 0, None
    shown = lambda: {c: v for c, v in sorted(count.items()) if v}
    m.step(f"Grow a window. Letters to repaint = size − the most common letter. That must stay ≤ `{k}`.",
           arr(chars), hmap(shown(), "count in window"), note(k=k, best=0))
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        size, most = right - left + 1, max(count.values())
        if size - most > k:
            m.step(f"Add `{ch}`: size {size}, most common letter ×{most}, so {size - most} to repaint. **Too many** (k = {k})!",
                   arr(chars, win=(left, right), hi={**grey_before(left), right: "bad"}, ptr={"L": left, "R": right}),
                   hmap(shown(), "count in window", hi={ch: "new"}), note(k=k, repaint=size - most, best=best))
        while (right - left + 1) - max(count.values()) > k:
            count[s[left]] -= 1
            left += 1
            size, most = right - left + 1, max(count.values())
            m.step(f"Drop `{s[left - 1]}` from the left. Now size {size} needs {size - most} repaint(s)." +
                   (" Still too many." if size - most > k else " OK again."),
                   arr(chars, win=(left, right), hi={**grey_before(left), left - 1: "swap"}, ptr={"L": left, "R": right}),
                   hmap(shown(), "count in window"), note(k=k, repaint=size - most, best=best))
        if size > best:
            best, best_at = size, (left, right)
            m.step(f"Add `{ch}`: `{s[left:right + 1]}` needs {size - most} repaint(s) ≤ {k}. Fine — a new best, **{best}**!",
                   arr(chars, win=(left, right), hi={**grey_before(left), right: "new"}, ptr={"L": left, "R": right}),
                   hmap(shown(), "count in window", hi={ch: "new"}), note(k=k, repaint=size - most, best=best))
        else:
            m.step(f"`{s[left:right + 1]}` needs {size - most} repaint(s) ≤ {k}. Fine, but not longer than {best}.",
                   arr(chars, win=(left, right), hi={**grey_before(left), right: "look"}, ptr={"L": left, "R": right}),
                   hmap(shown(), "count in window"), note(k=k, repaint=size - most, best=best))
    a, b = best_at
    m.step(f"Answer: **{best}**. Repaint `{s[a:b + 1]}` into one letter.",
           arr(chars, hi={i: "good" for i in range(a, b + 1)}), note(best=best))
    return best, m

s, k = "AABABBA", 1
answer, m = character_replacement_movie(s, k)
assert answer == character_replacement(s, k) == 4
m.show()

---

## 25 · Permutation in String · [LeetCode 567](https://leetcode.com/problems/permutation-in-string/) · 🟡 Medium

**🧸 Like you're five:** Your friend spelled a word with letter tiles and then mixed them up. Is that same bag of tiles lying
        somewhere side by side in a long row? Slide a frame exactly as wide as the word along the row and ask: "are
        these the same tiles?"

**The question:** Return `True` if some **permutation** (the same letters in any order) of `s1` appears as a substring of `s2`, otherwise `False`. Both are lowercase English letters.

| Input | Output |
|---|---|
| `s1 = "ab", s2 = "eidbaooo"` | `True` |
| `s1 = "ab", s2 = "eidboaoo"` | `False` |
| `s1 = "abc", s2 = "ab"` | `False` |

**🔎 How to spot it:** "**permutation / anagram** of p inside s" → the order does not matter, only the **counts**, and the
        piece always has the **same length** → a fixed-size window with letter counts.

**💡 The trick (fixed-size window + counts):** Count the letters of `s1` (`need`). Slide a window of exactly `len(s1)` letters over `s2`, keeping its
        counts up to date: add the letter that walks in on the right, take away the letter that walks out on the left.
        When the counts are equal, you found one. The trap: remove letters whose count drops to `0`, otherwise
        `{'a': 0}` and `{}` look different when you compare.

**⏱ Cost:** **O(26·n) = O(n)** time (comparing two counters of at most 26 letters per step) · **O(26)** space. Brute force: sort every substring of that length — O(n·k log k).

In [ ]:
def check_inclusion(s1, s2):
    k = len(s1)
    if k > len(s2):
        return False
    need, window = Counter(s1), Counter(s2[:k])
    if window == need:
        return True
    for right in range(k, len(s2)):
        window[s2[right]] += 1            # a new letter walks in on the right
        out = s2[right - k]
        window[out] -= 1                  # the oldest letter walks out on the left
        if window[out] == 0:
            del window[out]               # drop zeros so the two counters compare fairly
        if window == need:
            return True
    return False

assert check_inclusion("ab", "eidbaooo") is True
assert check_inclusion("ab", "eidboaoo") is False
assert check_inclusion("abc", "ab") is False     # s1 longer than s2
assert check_inclusion("a", "a") is True
assert check_inclusion("adc", "dcda") is True     # "cda"

rng = random.Random(0)
for _ in range(300):
    s1 = "".join(rng.choice("abc") for _ in range(rng.randint(1, 3)))
    s2 = "".join(rng.choice("abc") for _ in range(rng.randint(1, 8)))
    brute = any(sorted(s2[i:i + len(s1)]) == sorted(s1) for i in range(len(s2) - len(s1) + 1))
    assert check_inclusion(s1, s2) == brute
print(check_inclusion("ab", "eidbaooo"))

In [ ]:
def check_inclusion_movie(s1, s2):
    m = Movie("Permutation in String", f"Is some mix-up of {s1!r} hiding inside {s2!r}?")
    k, need, window, chars = len(s1), Counter(s1), Counter(), list(s2)
    tidy = lambda c: dict(sorted(c.items()))
    m.step(f"We need exactly these letters. Slide a window of size **{k}** and compare its counts with `need`.",
           arr(chars), hmap(tidy(need), "need (letters of s1)"), hmap({}, "window has"))
    for right, ch in enumerate(s2):
        window[ch] += 1
        if right >= k:
            out = s2[right - k]
            window[out] -= 1
            if window[out] == 0:
                del window[out]
        if right < k - 1:
            continue                                   # the first window is still filling up
        left = right - k + 1
        if window == need:
            m.step(f"Window `{s2[left:right + 1]}` has exactly the letters of `{s1}`. Found one → **True** ✅",
                   arr(chars, win=(left, right), hi={i: "good" for i in range(left, right + 1)}),
                   hmap(tidy(need), "need (letters of s1)", hi={c: "good" for c in need}),
                   hmap(tidy(window), "window has", hi={c: "good" for c in window}))
            return True, m
        diff = next(c for c in sorted(set(need) | set(window)) if need[c] != window[c])
        m.step(f"Window `{s2[left:right + 1]}`: we need {need[diff]} `{diff}` but it has {window[diff]}. Not a match — slide right.",
               arr(chars, win=(left, right), hi={**grey_before(left), right: "new"}),
               hmap(tidy(need), "need (letters of s1)", hi={diff: "bad"}),
               hmap(tidy(window), "window has", hi={diff: "bad"}))
    m.step("The window reached the end and never matched → **False**.", arr(chars), hmap(tidy(need), "need"))
    return False, m

s1, s2 = "abc", "ccbxbcad"
answer, m = check_inclusion_movie(s1, s2)
assert answer == check_inclusion(s1, s2) is True
m.show()

---

## 26 · Minimum Size Subarray Sum · [LeetCode 209](https://leetcode.com/problems/minimum-size-subarray-sum/) · 🟡 Medium

**🧸 Like you're five:** You need at least 7 sweets, and the sweets sit in jars in a row. You may only take jars that stand
        next to each other. What is the **fewest jars** that give you enough? Add jars on the right until you have
        enough, then try giving jars back from the left.

**The question:** Given **positive** integers `nums` and a positive `target`, return the length of the shortest subarray whose sum is **at least** `target`, or `0` if there is none.

| Input | Output |
|---|---|
| `target = 7, nums = [2, 3, 1, 2, 4, 3]` | `2` |
| `target = 4, nums = [1, 4, 4]` | `1` |
| `target = 11, nums = [1, 1, 1, 1, 1, 1, 1, 1]` | `0` |

**🔎 How to spot it:** "**shortest / longest subarray** with sum ≥ / ≤ something" and all numbers are **positive** → sliding
        window. (With negative numbers the window trick breaks — use prefix sums instead.)

**💡 The trick (sliding window):** Grow the window by adding `nums[right]` to `total`. **While** `total >= target`, the window is good:
        record its length, then shrink it by taking `nums[left]` away and moving `left` on. It must be a `while`, not
        an `if` — after one shrink the window may still be big enough. Return `0` if no window was ever good.

**⏱ Cost:** **O(n)** time (each number is added once and removed at most once) · **O(1)** space. Brute force: every start with every end — O(n²).

In [ ]:
def min_sub_array_len(target, nums):
    left = total = 0
    best = float("inf")
    for right, x in enumerate(nums):
        total += x                             # grow: take the next number in
        while total >= target:                 # big enough? then try to make it shorter
            best = min(best, right - left + 1)
            total -= nums[left]                # shrink from the left
            left += 1
    return 0 if best == float("inf") else best

assert min_sub_array_len(7, [2, 3, 1, 2, 4, 3]) == 2       # [4, 3]
assert min_sub_array_len(4, [1, 4, 4]) == 1
assert min_sub_array_len(11, [1, 1, 1, 1, 1, 1, 1, 1]) == 0  # never enough
assert min_sub_array_len(15, [1, 2, 3, 4, 5]) == 5         # needs everything
assert min_sub_array_len(3, [5]) == 1

rng = random.Random(0)
for _ in range(300):
    ns, t = [rng.randint(1, 6) for _ in range(rng.randint(1, 9))], rng.randint(1, 20)
    lens = [j - i for i in range(len(ns)) for j in range(i + 1, len(ns) + 1) if sum(ns[i:j]) >= t]
    assert min_sub_array_len(t, ns) == min(lens, default=0)
print(min_sub_array_len(7, [2, 3, 1, 2, 4, 3]))

In [ ]:
def min_sub_array_len_movie(target, nums):
    m = Movie("Minimum Size Subarray Sum", f"Fewest boxes in a row that add up to at least {target}")
    left = total = 0
    best, best_at = float("inf"), None
    m.step(f"Grow the window on the right until the total reaches **{target}**, then shrink it from the left.",
           arr(nums), note(total=0, target=target, best=best))
    for right, x in enumerate(nums):
        total += x
        if total < target:
            m.step(f"Add `{x}`: total is `{total}`, less than {target}. Grow more.",
                   arr(nums, win=(left, right), hi={**grey_before(left), right: "new"}, ptr={"L": left, "R": right}),
                   note(total=total, target=target, best=best))
        first = True
        while total >= target:
            size = right - left + 1
            new_best = size < best
            best = min(best, size)
            if new_best:
                best_at = (left, right)
            start = f"Add `{x}`: total `{total}`" if first else f"Total `{total}`"
            m.step(f"{start} ≥ {target}! Length {size}" + (" — a new best." if new_best else ".") +
                   f" Try shorter: drop `{nums[left]}` from the left.",
                   arr(nums, win=(left, right), hi={**grey_before(left), left: "swap", right: "new" if first else "win"},
                       ptr={"L": left, "R": right}), note(total=total, target=target, best=best))
            total -= nums[left]
            left += 1
            first = False
    a, b = best_at
    m.step(f"Answer: **{best}** boxes, `{nums[a:b + 1]}`, add up to {sum(nums[a:b + 1])}.",
           arr(nums, hi={i: "good" for i in range(a, b + 1)}), note(best=best))
    return best, m

target, nums = 7, [2, 3, 1, 2, 4, 3]
answer, m = min_sub_array_len_movie(target, nums)
assert answer == min_sub_array_len(target, nums) == 2
m.show()

---

## 27 · Find All Anagrams in a String · [LeetCode 438](https://leetcode.com/problems/find-all-anagrams-in-a-string/) · 🟡 Medium

**🧸 Like you're five:** The same game as problem 25, but now you shout out **every** place where the mixed-up tiles appear,
        not just the first one.

**The question:** Given strings `s` and `p` (lowercase letters), return the start indices of every substring of `s` that is an **anagram** of `p` (the same letters, any order). Any order of the answer is accepted; we return them from left to right.

| Input | Output |
|---|---|
| `s = "cbaebabacd", p = "abc"` | `[0, 6]` |
| `s = "abab", p = "ab"` | `[0, 1, 2]` |
| `s = "a", p = "ab"` | `[]` |

**🔎 How to spot it:** "**all** anagrams / permutations", "return every start index" → the fixed-size window of problem 25,
        but keep going after a match and collect the starts.

**💡 The trick (fixed-size window + counts):** Count `p`. Walk `right` over `s`, adding `s[right]` to the window counts; once the window is longer
        than `len(p)`, take `s[right − len(p)]` away (and delete zero counts). Whenever the window is full and the
        counts are equal, record `right − len(p) + 1`. Windows may **overlap** (`"abab"` → `[0, 1, 2]`), so never jump
        ahead after a match.

**⏱ Cost:** **O(26·n) = O(n)** time · **O(26)** space. Brute force: sort every substring of length `len(p)` — O(n·k log k).

In [ ]:
def find_anagrams(s, p):
    k, need, window, out = len(p), Counter(p), Counter(), []
    for right, ch in enumerate(s):
        window[ch] += 1                        # the new letter walks in
        if right >= k:
            old = s[right - k]
            window[old] -= 1                   # the oldest letter walks out
            if window[old] == 0:
                del window[old]
        if right >= k - 1 and window == need:  # the window is full and matches
            out.append(right - k + 1)
    return out

assert find_anagrams("cbaebabacd", "abc") == [0, 6]
assert find_anagrams("abab", "ab") == [0, 1, 2]     # windows overlap
assert find_anagrams("a", "ab") == []               # p longer than s
assert find_anagrams("aaaa", "aa") == [0, 1, 2]

rng = random.Random(0)
for _ in range(300):
    p = "".join(rng.choice("ab") for _ in range(rng.randint(1, 3)))
    s = "".join(rng.choice("abc") for _ in range(rng.randint(0, 9)))
    brute = [i for i in range(len(s) - len(p) + 1) if sorted(s[i:i + len(p)]) == sorted(p)]
    assert find_anagrams(s, p) == brute
print(find_anagrams("cbaebabacd", "abc"))

In [ ]:
def find_anagrams_movie(s, p):
    m = Movie("Find All Anagrams in a String", f"Every place where the letters of {p!r} sit side by side")
    k, need, window, out, chars = len(p), Counter(p), Counter(), [], list(s)
    tidy = lambda c: dict(sorted(c.items()))
    m.step(f"Slide a window of size **{k}** along the row. Each time its counts equal `need`, write down where it starts.",
           arr(chars), hmap(tidy(need), "need (letters of p)"), hmap({}, "window has"), note(found=out))
    for right, ch in enumerate(s):
        window[ch] += 1
        if right >= k:
            old = s[right - k]
            window[old] -= 1
            if window[old] == 0:
                del window[old]
        if right < k - 1:
            continue
        left = right - k + 1
        if window == need:
            out.append(left)
            m.step(f"Window `{s[left:right + 1]}` uses the same letters as `{p}`. Write down its start, **{left}** ✅",
                   arr(chars, win=(left, right), hi={i: "good" for i in range(left, right + 1)}),
                   hmap(tidy(need), "need (letters of p)"), hmap(tidy(window), "window has", hi={c: "good" for c in window}),
                   note(found=out))
        else:
            diff = next(c for c in sorted(set(need) | set(window)) if need[c] != window[c])
            m.step(f"Window `{s[left:right + 1]}`: we need {need[diff]} `{diff}` but it has {window[diff]}. Slide on.",
                   arr(chars, win=(left, right), hi={right: "new"}),
                   hmap(tidy(need), "need (letters of p)", hi={diff: "bad"}),
                   hmap(tidy(window), "window has", hi={diff: "bad"}), note(found=out))
    m.step(f"Answer: **{out}** — the windows starting there are anagrams of `{p}`.",
           arr(chars, hi={i: "good" for a in out for i in range(a, a + k)}, ptr={f"#{n + 1}": a for n, a in enumerate(out)}),
           note(found=out))
    return out, m

s, p = "cbaebabacd", "abc"
answer, m = find_anagrams_movie(s, p)
assert answer == find_anagrams(s, p) == [0, 6]
m.show()

---

## 28 · Valid Parentheses · [LeetCode 20](https://leetcode.com/problems/valid-parentheses/) · 🟢 Easy

**🧸 Like you're five:** Every open door `(` `[` `{` must be shut by the matching door, and the **last** door you opened must be
        the **first** one you shut. Put each open door on a pile; a closing door must match the top of the pile.

**The question:** A string `s` contains only `()[]{}`. It is valid if every opener is closed by the same type of bracket, in the right order, and every closer has an opener. Return `True` or `False`.

| Input | Output |
|---|---|
| `s = "()[]{}"` | `True` |
| `s = "([)]"` | `False` |
| `s = "{[]}"` | `True` |

**🔎 How to spot it:** "**matching / nested** pairs", "balanced", "the most recent unmatched thing" → a **stack**. The
        newest opener must be closed first — that is exactly last-in-first-out.

**💡 The trick (stack):** Walk the string. An opener goes on the stack. A closer must find **its own** opener on top: if the
        stack is empty or the top is a different kind, return `False`; otherwise pop. At the end the stack must be
        **empty** — people forget this check, and `"(("` sneaks through as valid.

**⏱ Cost:** **O(n)** time · **O(n)** space for the stack. Brute force: keep deleting `()`, `[]`, `{}` pairs until nothing changes — O(n²).

In [ ]:
def is_valid(s):
    pairs = {")": "(", "]": "[", "}": "{"}      # closer -> the opener it needs
    stack = []
    for ch in s:
        if ch in pairs:                           # a closer
            if not stack or stack[-1] != pairs[ch]:
                return False                      # nothing to close, or the wrong kind on top
            stack.pop()
        else:
            stack.append(ch)                      # an opener waits on the pile
    return not stack                              # leftovers = openers that were never closed

assert is_valid("()") and is_valid("()[]{}") and is_valid("{[]}")
assert not is_valid("(]")
assert not is_valid("([)]")        # crossed, not nested
assert not is_valid("((")          # the empty-stack check at the end
assert not is_valid(")")           # a closer with nothing to close

def brute(s):
    while any(pair in s for pair in ("()", "[]", "{}")):
        s = s.replace("()", "").replace("[]", "").replace("{}", "")
    return s == ""

rng = random.Random(0)
for _ in range(500):
    s = "".join(rng.choice("()[]{}") for _ in range(rng.randint(1, 8)))
    assert is_valid(s) == brute(s)
print(is_valid("()[]{}"))

In [ ]:
def is_valid_movie(s):
    m = Movie("Valid Parentheses", "Does every opener get closed by the right closer, in order?")
    pairs, st, chars = {")": "(", "]": "[", "}": "{"}, [], list(s)
    m.step("Openers go on a pile. A closer must match the **top** plate of the pile.", arr(chars), stack(st, "openers waiting"))
    for i, ch in enumerate(s):
        past = grey_before(i)
        if ch in pairs:
            if not st or st[-1] != pairs[ch]:
                top = f"`{st[-1]}`" if st else "nothing"
                m.step(f"`{ch}` needs `{pairs[ch]}` on top, but the top is {top}. Not valid → **False**.",
                       arr(chars, hi={**past, i: "bad"}, ptr={"i": i}), stack(st, "openers waiting", hi={len(st) - 1: "bad"}))
                return False, m
            m.step(f"`{ch}` closes the `{st[-1]}` on top. A match! Take that plate off.",
                   arr(chars, hi={**past, i: "good"}, ptr={"i": i}), stack(st, "openers waiting", hi={len(st) - 1: "good"}))
            st.pop()
        else:
            st.append(ch)
            m.step(f"`{ch}` is an opener. Put it on the pile to wait for its partner.",
                   arr(chars, hi={**past, i: "new"}, ptr={"i": i}), stack(st, "openers waiting", hi={len(st) - 1: "new"}))
    if st:
        m.step(f"The end, but `{st[-1]}` still waits on the pile. Never closed → **False**.",
               arr(chars, hi=grey_before(len(s))), stack(st, "openers waiting", hi={len(st) - 1: "bad"}))
        return False, m
    m.step("The end, and the pile is empty: every opener was closed → **True** ✅",
           arr(chars, hi={i: "good" for i in range(len(s))}), stack(st, "openers waiting"))
    return True, m

s = "([{}])()"
answer, m = is_valid_movie(s)
assert answer == is_valid(s) is True
m.show()

---

## 29 · Min Stack · [LeetCode 155](https://leetcode.com/problems/min-stack/) · 🟡 Medium

**🧸 Like you're five:** You have a pile of plates, each with a number. Next to it you build a second pile: each time you add a
        plate, you also add a sticky note saying "the smallest plate so far". The top sticky note always knows the
        answer — no searching.

**The question:** Design a stack with `push(val)`, `pop()`, `top()` and `getMin()` (the smallest element in the stack), each in **O(1)** time. `pop`, `top` and `getMin` are only called on a non-empty stack.

| Input | Output |
|---|---|
| `push(-2), push(0), push(-3), getMin(), pop(), top(), getMin()` | `[-3, 0, -2]` |
| `push(5), getMin()` | `[5]` |
| `push(1), push(1), pop(), getMin()` | `[1]` |

**🔎 How to spot it:** "design a stack / queue that **also** answers X in **O(1)**" → keep a second, parallel structure that
        remembers X for every height of the stack.

**💡 The trick (two stacks):** Keep two lists that always have the same length: `stack` and `mins`, where `mins[i]` is the smallest of
        `stack[0..i]`. `push(x)` appends `x` and `min(x, mins[-1])`. `pop()` pops both. `getMin()` is `mins[-1]`.
        The trap: keeping just **one** `min` variable — after popping the smallest you would not know the next
        smallest.

**⏱ Cost:** **O(1)** time for every operation · **O(n)** space (two lists). Brute force: `min(stack)` on every `getMin` — O(n).

In [ ]:
class MinStack:
    def __init__(self):
        self.stack = []        # the real plates
        self.mins = []         # mins[i] = the smallest of stack[0..i]

    def push(self, val):
        self.stack.append(val)
        self.mins.append(val if not self.mins else min(val, self.mins[-1]))

    def pop(self):
        self.stack.pop()
        self.mins.pop()        # the two piles always have the same height

    def top(self):
        return self.stack[-1]

    def getMin(self):          # LeetCode calls it getMin
        return self.mins[-1]

ms = MinStack()
ms.push(-2); ms.push(0); ms.push(-3)
assert ms.getMin() == -3
ms.pop()
assert ms.top() == 0 and ms.getMin() == -2

ms = MinStack()
ms.push(1); ms.push(1); ms.pop()
assert ms.getMin() == 1          # duplicates of the minimum are safe

rng = random.Random(0)
for _ in range(200):
    ms, plain = MinStack(), []
    for _ in range(20):
        if plain and rng.random() < 0.4:
            ms.pop(); plain.pop()
        else:
            x = rng.randint(-5, 5); ms.push(x); plain.append(x)
        if plain:
            assert ms.getMin() == min(plain) and ms.top() == plain[-1]
print(ms.stack, ms.mins)

In [ ]:
def min_stack_movie(ops):
    m = Movie("Min Stack", "A pile of plates that always knows its smallest plate")
    ms, out = MinStack(), []
    piles = lambda hi=None: (stack(ms.stack, "plates", hi), stack(ms.mins, "smallest so far", hi))
    m.step("Two piles of the same height: the plates, and beside each plate **the smallest plate so far**.", *piles())
    for op, *arg in ops:
        top = len(ms.stack) - 1
        if op == "push":
            ms.push(arg[0])
            why = "the only plate, so it is also the smallest" if len(ms.stack) == 1 else f"min({arg[0]}, {ms.mins[-2]}) = {ms.mins[-1]}"
            m.step(f"`push({arg[0]})`: add the plate, and beside it the smallest so far: {why}.", *piles({top + 1: "new"}))
        elif op == "pop":
            m.step(f"`pop()`: take off the top plate `{ms.stack[-1]}` **and** its sticky note `{ms.mins[-1]}`.", *piles({top: "swap"}))
            ms.pop()
        elif op == "getMin":
            out.append(ms.getMin())
            m.step(f"`getMin()`: just read the top sticky note → **{out[-1]}**. No searching!",
                   stack(ms.stack, "plates"), stack(ms.mins, "smallest so far", {top: "good"}))
        else:
            out.append(ms.top())
            m.step(f"`top()`: the top plate is **{out[-1]}**.", stack(ms.stack, "plates", {top: "good"}), stack(ms.mins, "smallest so far"))
    m.step(f"The answers, in order: **{out}**. Every one took a single look.", *piles({len(ms.stack) - 1: "good"}), note(answers=out))
    return out, m

ops = [("push", 5), ("push", 3), ("push", 7), ("push", 2), ("getMin",), ("pop",), ("getMin",), ("pop",), ("pop",), ("getMin",), ("top",)]
answer, m = min_stack_movie(ops)
plain, expected = [], []                        # the slow way: search the whole pile
for op, *arg in ops:
    if op == "push": plain.append(arg[0])
    elif op == "pop": plain.pop()
    else: expected.append(min(plain) if op == "getMin" else plain[-1])
assert answer == expected == [2, 3, 5, 5]
m.show()

---

## 30 · Evaluate Reverse Polish Notation · [LeetCode 150](https://leetcode.com/problems/evaluate-reverse-polish-notation/) · 🟡 Medium

**🧸 Like you're five:** A calculator that reads numbers first and the sign after: `3 4 +` means 3 + 4. Numbers wait on a
        pile. When a sign arrives, it grabs the two top numbers, does its sum, and puts the result back on the pile.

**The question:** Evaluate an arithmetic expression given as a list of tokens in Reverse Polish Notation. Operators are `+ - * /`; division **truncates toward zero**. The expression is always valid and every result fits in 32 bits.

| Input | Output |
|---|---|
| `tokens = ["2", "1", "+", "3", "*"]` | `9` |
| `tokens = ["4", "13", "5", "/", "+"]` | `6` |
| `tokens = ["6", "-4", "/"]` | `-1` |

**🔎 How to spot it:** "**postfix / reverse Polish**", "evaluate an expression", "the operator applies to the last two
        results" → a stack of numbers.

**💡 The trick (stack):** A number → push it. An operator → pop **`b` first, then `a`** (the top is the *second* number), push
        `a op b`. At the end the only number on the stack is the answer. Two traps: the order for `-` and `/`, and
        Python's `//` rounds **down** (`-3 // 2 == -2`) — LeetCode wants rounding toward zero, so use `int(a / b)`.
        Also `"-4"` is a number, not a minus sign, so test `t in "+-*/"` on the whole token.

**⏱ Cost:** **O(n)** time · **O(n)** space for the stack. There is no simpler brute force — the stack *is* the direct method.

In [ ]:
def eval_rpn(tokens):
    stack = []
    for t in tokens:
        if t in {"+", "-", "*", "/"}:
            b = stack.pop()                 # the top plate is the SECOND number
            a = stack.pop()
            if t == "+":
                stack.append(a + b)
            elif t == "-":
                stack.append(a - b)
            elif t == "*":
                stack.append(a * b)
            else:
                stack.append(int(a / b))    # divide, then chop toward zero (-3.5 -> -3)
        else:
            stack.append(int(t))            # "-4" is a number, not a minus sign
    return stack[0]

assert eval_rpn(["2", "1", "+", "3", "*"]) == 9
assert eval_rpn(["4", "13", "5", "/", "+"]) == 6
assert eval_rpn(["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]) == 22
assert eval_rpn(["6", "-4", "/"]) == -1      # toward zero, not -2
assert eval_rpn(["3", "5", "-"]) == -2       # a - b, not b - a
assert eval_rpn(["7"]) == 7
print(eval_rpn(["4", "13", "5", "/", "+"]))

In [ ]:
def eval_rpn_movie(tokens):
    m = Movie("Evaluate Reverse Polish Notation", "Numbers wait on a pile; a sign eats the top two")
    st = []
    m.step("Read the tokens left to right. Numbers go on the pile. A sign takes the **top two** numbers.",
           arr(tokens), stack(st, "numbers waiting"))
    for i, t in enumerate(tokens):
        past = grey_before(i)
        if t in {"+", "-", "*", "/"}:
            b, a = st[-1], st[-2]
            m.step(f"`{t}` is a sign: take the top `{b}` (the **second** number) and `{a}` under it (the first).",
                   arr(tokens, hi={**past, i: "look"}, ptr={"i": i}), stack(st, "numbers waiting", {len(st) - 1: "swap", len(st) - 2: "swap"}))
            st.pop(); st.pop()
            val = a + b if t == "+" else a - b if t == "-" else a * b if t == "*" else int(a / b)
            extra = f" That is `{a / b:g}`, chopped toward zero → `{val}` (not `{a // b}`)." if t == "/" and a / b != val else ""
            st.append(val)
            m.step(f"`{a} {t} {b}` = `{val}`. Put the result back on the pile.{extra}",
                   arr(tokens, hi={**past, i: "swap"}, ptr={"i": i}), stack(st, "numbers waiting", {len(st) - 1: "new"}))
        else:
            st.append(int(t))
            m.step(f"`{t}` is a number: put it on the pile.",
                   arr(tokens, hi={**past, i: "new"}, ptr={"i": i}), stack(st, "numbers waiting", {len(st) - 1: "new"}))
    m.step(f"All tokens used. The one plate left is the answer: **{st[0]}** ✅",
           arr(tokens, hi=grey_before(len(tokens))), stack(st, "numbers waiting", {0: "good"}))
    return st[0], m

tokens = ["7", "-2", "/", "4", "*", "5", "+"]
answer, m = eval_rpn_movie(tokens)
assert answer == eval_rpn(tokens) == -7
m.show()

---

## 31 · Daily Temperatures · [LeetCode 739](https://leetcode.com/problems/daily-temperatures/) · 🟡 Medium

**🧸 Like you're five:** Each cold day waits in a line for a warmer day. When a warm day comes, every waiting day that is colder
        gets its answer and leaves. The waiting days always get warmer from top to bottom of the pile, so we only ever
        check the top.

**The question:** Given daily `temperatures`, return `answer` where `answer[i]` is how many days after day `i` you must wait for a **warmer** temperature, or `0` if none comes.

| Input | Output |
|---|---|
| `temperatures = [73, 74, 75, 71, 69, 72, 76, 73]` | `[1, 1, 4, 2, 1, 1, 0, 0]` |
| `temperatures = [30, 60, 90]` | `[1, 1, 0]` |
| `temperatures = [70, 70]` | `[0, 0]` |

**🔎 How to spot it:** "the **next greater / warmer / taller** element", "how far to the next bigger one" → a **monotonic
        stack**: a stack of the items that are still waiting, kept in order.

**💡 The trick (monotonic stack):** Keep a stack of **indices** of days still waiting. For each new day `i`: **while** it is warmer than
        the day on top, pop that day `j` and set `answer[j] = i − j`. Then push `i`. Days never popped keep `0`.
        Store indices, not temperatures — you need the index to compute the distance. Equal is not warmer: use `<`.

**⏱ Cost:** **O(n)** time (each day is pushed once and popped at most once) · **O(n)** space. Brute force: for each day scan forward — O(n²).

In [ ]:
def daily_temperatures(temperatures):
    answer = [0] * len(temperatures)
    stack = []                                   # indices of days still waiting for a warmer day
    for i, t in enumerate(temperatures):
        while stack and temperatures[stack[-1]] < t:   # today is warmer than the top waiting day
            j = stack.pop()
            answer[j] = i - j                          # day j waited i - j days
        stack.append(i)                          # today waits too
    return answer

assert daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0]
assert daily_temperatures([30, 40, 50, 60]) == [1, 1, 1, 0]
assert daily_temperatures([30, 60, 90]) == [1, 1, 0]
assert daily_temperatures([70, 70]) == [0, 0]     # equal is not warmer
assert daily_temperatures([50]) == [0]

rng = random.Random(0)
for _ in range(300):
    ts = [rng.randint(30, 40) for _ in range(rng.randint(1, 9))]
    brute = [next((j - i for j in range(i + 1, len(ts)) if ts[j] > ts[i]), 0) for i in range(len(ts))]
    assert daily_temperatures(ts) == brute
print(daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]))

In [ ]:
def daily_temperatures_movie(temps):
    m = Movie("Daily Temperatures", "How many days until a warmer day?")
    ans, st, answered = [0] * len(temps), [], set()
    pile = lambda hi=None: stack([f"{temps[j]} (day {j})" for j in st], "days still waiting", hi)
    days = lambda i, role: arr(temps, "temperature", hi={**{j: "done" for j in answered}, i: role}, ptr={"today": i})
    m.step("Days that have not met a warmer day yet wait on a pile. The coldest one is always on top.",
           arr(temps, "temperature"), pile(), arr(ans, "answer: days to wait"))
    for i, t in enumerate(temps):
        while st and temps[st[-1]] < t:
            j = st[-1]
            ans[j] = i - j
            m.step(f"Day {i} is `{t}`, warmer than the top plate `{temps[j]}` (day {j}). Day {j} waited **{i - j}** day(s). Pop it.",
                   days(i, "look"), pile({len(st) - 1: "good"}), arr(ans, "answer: days to wait", hi={j: "new"}))
            st.pop()
            answered.add(j)
        below = f"It is not warmer than `{temps[st[-1]]}` below, so it" if st else "The pile is empty, so it"
        st.append(i)
        m.step(f"Day {i} (`{t}`) waits for a warmer day too. {below} goes on top.",
               days(i, "new"), pile({len(st) - 1: "new"}), arr(ans, "answer: days to wait"))
    m.step(f"Done. Days still on the pile never got warmer, so they keep `0`. Answer: **{ans}**",
           arr(temps, "temperature"), pile(), arr(ans, "answer: days to wait", hi={k: "good" for k in range(len(ans))}))
    return ans, m

temps = [73, 74, 75, 71, 69, 72, 76, 73]
answer, m = daily_temperatures_movie(temps)
assert answer == daily_temperatures(temps) == [1, 1, 4, 2, 1, 1, 0, 0]
m.show()

---

## 32 · Decode String · [LeetCode 394](https://leetcode.com/problems/decode-string/) · 🟡 Medium

**🧸 Like you're five:** `2[a3[b]c]` is a present inside a present. To unwrap it you must open the **inner** box first. When you
        start a new box, you put what you had so far on a pile; when the box closes, you take it back off the pile and
        glue the unwrapped box onto it.

**The question:** Decode a string where `k[encoded]` means `encoded` repeated `k` times. Brackets can be nested, `k` is a positive integer (maybe several digits), and the input is always valid.

| Input | Output |
|---|---|
| `s = "3[a]2[bc]"` | `"aaabcbc"` |
| `s = "3[a2[c]]"` | `"accaccacc"` |
| `s = "2[abc]3[cd]ef"` | `"abcabccdcdcdef"` |

**🔎 How to spot it:** "**nested** brackets", "decode / evaluate from the inside out" → a stack that saves the work you
        were doing before each `[` (or recursion, which uses the call stack for the same job).

**💡 The trick (stack):** Keep `cur` (the text being built) and `num` (the repeat count being read). Digit → `num = num·10 +
        digit` (the trap: `12[a]` has **two** digits). `[` → push `(cur, num)` and start fresh. Letter → add to `cur`.
        `]` → pop `(before, times)` and set `cur = before + cur·times`. The answer is `cur` at the end.

**⏱ Cost:** **O(output length)** time and space — every letter of the answer is written once per level it is copied through. No simpler brute force exists.

In [ ]:
def decode_string(s):
    stack = []                       # saved (text before the '[', how many times to repeat)
    cur, num = "", 0
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)            # numbers can have many digits: "12[a]"
        elif ch == "[":
            stack.append((cur, num))            # park what we have, start fresh inside the box
            cur, num = "", 0
        elif ch == "]":
            before, times = stack.pop()
            cur = before + cur * times          # glue the repeated inside onto what came before
        else:
            cur += ch
    return cur

assert decode_string("3[a]2[bc]") == "aaabcbc"
assert decode_string("3[a2[c]]") == "accaccacc"
assert decode_string("2[abc]3[cd]ef") == "abcabccdcdcdef"
assert decode_string("abc") == "abc"            # no brackets at all
assert decode_string("10[a]") == "a" * 10       # a two-digit number
assert decode_string("2[a3[b]c]") == "abbbcabbbc"
print(decode_string("3[a2[c]]"))

In [ ]:
def decode_string_movie(s):
    m = Movie("Decode String", "Unwrap k[...] boxes, the inner box first")
    st, cur, num, chars = [], "", 0, list(s)
    pile = lambda hi=None: stack([f"{b!r} ×{t}" for b, t in st], "parked (text before, times)", hi)
    m.step("`cur` is the text we are building, `num` the repeat count. At `[` we park both on the pile.",
           arr(chars), pile(), note(cur=repr(cur), num=num))
    for i, ch in enumerate(s):
        past, hi = grey_before(i), None
        if ch.isdigit():
            num = num * 10 + int(ch)
            cap, role = f"`{ch}` is a digit: the next box repeats **{num}** times.", "look"
        elif ch == "[":
            st.append((cur, num))
            cap, role, hi = f"`[` opens a box. Park `{cur!r}` and `{num}` on the pile; start a fresh, empty `cur`.", "new", {len(st) - 1: "new"}
            cur, num = "", 0
        elif ch == "]":
            before, times = st[-1]
            m.step(f"`]` closes the box. Take `{before!r} ×{times}` off the top of the pile.",
                   arr(chars, hi={**past, i: "swap"}, ptr={"i": i}), pile({len(st) - 1: "swap"}), note(cur=repr(cur), num=num))
            st.pop()
            inner, cur = cur, before + cur * times
            cap, role = f"Glue: `{before!r}` + `{inner!r}` × {times} = `{cur!r}`.", "good"
        else:
            cur += ch
            cap, role = f"`{ch}` is a letter: add it to `cur`.", "look"
        m.step(cap, arr(chars, hi={**past, i: role}, ptr={"i": i}), pile(hi), note(cur=repr(cur), num=num))
    m.step(f"Everything is unwrapped: **{cur}** ✅", arr(list(cur), "answer", hi={k: "good" for k in range(len(cur))}), pile())
    return cur, m

s = "2[a3[b]c]"
answer, m = decode_string_movie(s)
assert answer == decode_string(s) == "abbbcabbbc"
m.show()

---

## 33 · Implement Queue using Stacks · [LeetCode 232](https://leetcode.com/problems/implement-queue-using-stacks/) · 🟢 Easy

**🧸 Like you're five:** A queue is a line of kids: first in, first out. A stack is a pile of plates: last in, first out. Pour a
        pile of plates onto a second pile and the order **flips** — the bottom plate lands on top. Two flips make a
        line!

**The question:** Build a first-in-first-out queue with `push(x)`, `pop()`, `peek()` and `empty()` using only stack operations (push to top, pop/peek from top, size, is empty). Aim for **amortised O(1)** per operation.

| Input | Output |
|---|---|
| `push(1), push(2), peek(), pop(), empty()` | `[1, 1, False]` |
| `push(1), pop(), empty()` | `[1, True]` |
| `push(1), push(2), pop(), push(3), pop(), pop()` | `[1, 2, 3]` |

**🔎 How to spot it:** "implement X using Y" → make Y's weakness disappear by **combining two** of them. Reversing twice
        restores the order.

**💡 The trick (two stacks):** Keep an `inbox` stack for pushes and an `outbox` stack for pops. To `pop` or `peek`: if `outbox` is
        **empty**, pour everything from `inbox` into it (this flips the order, so the oldest item ends on top); then
        use the top of `outbox`. The trap: pouring when `outbox` is **not** empty mixes the order up. Each item is
        moved at most once, so the average cost is O(1).

**⏱ Cost:** **Amortised O(1)** per operation (each item is pushed and popped at most twice in its life) · **O(n)** space. The naive way pours back and forth on every call — O(n) per operation.

In [ ]:
class MyQueue:
    def __init__(self):
        self.inbox = []        # new items are pushed here
        self.outbox = []       # items leave from here; the oldest is on top

    def push(self, x):
        self.inbox.append(x)

    def _pour(self):
        if not self.outbox:                # only pour when the outbox is empty!
            while self.inbox:
                self.outbox.append(self.inbox.pop())

    def pop(self):
        self._pour()
        return self.outbox.pop()

    def peek(self):
        self._pour()
        return self.outbox[-1]

    def empty(self):
        return not self.inbox and not self.outbox

q = MyQueue()
q.push(1); q.push(2)
assert q.peek() == 1 and q.pop() == 1 and q.empty() is False
q.push(3)
assert q.pop() == 2 and q.pop() == 3 and q.empty() is True

rng = random.Random(0)
for _ in range(200):
    q, line = MyQueue(), deque()
    for step in range(30):
        if line and rng.random() < 0.45:
            assert q.peek() == line[0] and q.pop() == line.popleft()
        else:
            q.push(step); line.append(step)
        assert q.empty() == (not line)
print("ok")

In [ ]:
def my_queue_movie(ops):
    m = Movie("Implement Queue using Stacks", "Two piles of plates pretending to be a line of kids")
    q, out = MyQueue(), []
    def pics(in_hi=None, out_hi=None):
        line = list(reversed(q.outbox)) + q.inbox           # the order the queue promises
        return (stack(q.inbox, "in (push here)", in_hi), stack(q.outbox, "out (pop from here)", out_hi),
                queue(line, "the line we pretend to be"))
    m.step("Pushes go on the **in** pile. Pops come from the **out** pile.", *pics())
    for op, *arg in ops:
        if op in ("pop", "peek") and not q.outbox and q.inbox:
            q._pour()
            m.step("The **out** pile is empty, so pour every plate across. Pouring flips them: the oldest lands on top.",
                   *pics(out_hi={k: "swap" for k in range(len(q.outbox))}))
        if op == "push":
            q.push(arg[0])
            m.step(f"`push({arg[0]})`: put it on the **in** pile. The out pile is left alone.", *pics(in_hi={len(q.inbox) - 1: "new"}))
        elif op == "peek":
            out.append(q.peek())
            m.step(f"`peek()`: the top of **out** is the oldest → **{out[-1]}**.", *pics(out_hi={len(q.outbox) - 1: "good"}))
        elif op == "pop":
            top = len(q.outbox) - 1
            m.step(f"`pop()`: take the top of **out**, `{q.outbox[-1]}` — the one that came first.", *pics(out_hi={top: "good"}))
            out.append(q.pop())
        else:
            out.append(q.empty())
            m.step(f"`empty()`: both piles are empty → **{out[-1]}**.", *pics())
    m.step(f"The answers, in order: **{out}** — first in, first out ✅", *pics(), note(answers=out))
    return out, m

ops = [("push", 1), ("push", 2), ("push", 3), ("peek",), ("pop",), ("push", 4), ("pop",), ("pop",), ("pop",), ("empty",)]
answer, m = my_queue_movie(ops)
line, expected = deque(), []                    # a real line of kids, for checking
for op, *arg in ops:
    if op == "push": line.append(arg[0])
    elif op == "peek": expected.append(line[0])
    elif op == "pop": expected.append(line.popleft())
    else: expected.append(not line)
assert answer == expected == [1, 1, 2, 3, 4, True]
m.show()

---

## 34 · Binary Search · [LeetCode 704](https://leetcode.com/problems/binary-search/) · 🟢 Easy

**🧸 Like you're five:** "I'm thinking of a number between 1 and 100." You guess 50. "Higher!" Now you know it is not 1 to 50 —
        half the numbers are gone with one guess. Always guess the middle of what is left.

**The question:** Given a **sorted** list of distinct integers `nums` and a `target`, return the index of `target`, or `-1` if it is not there. It must run in **O(log n)** time.

| Input | Output |
|---|---|
| `nums = [-1, 0, 3, 5, 9, 12], target = 9` | `4` |
| `nums = [-1, 0, 3, 5, 9, 12], target = 2` | `-1` |
| `nums = [5], target = 5` | `0` |

**🔎 How to spot it:** "**sorted**" + "find" + "O(log n)" → binary search. Any time you can say "if the answer is not
        here, I know which **side** it is on", you can halve.

**💡 The trick (binary search):** Keep `left` and `right`, the ends of the part that could still hold the target. Look at `mid = (left +
        right) // 2`. Equal → done. Too small → `left = mid + 1`. Too big → `right = mid − 1`. Stop when `left >
        right`. The traps: writing `left < right` (misses a one-item range) and `left = mid` (can loop forever).

**⏱ Cost:** **O(log n)** time (each guess halves the range: 1,000,000 items need about 20 guesses) · **O(1)** space. Brute force: check every item — O(n).

In [ ]:
def search(nums, target):
    left, right = 0, len(nums) - 1             # the target, if anywhere, is in nums[left..right]
    while left <= right:                       # <=, so a one-item range is still checked
        mid = (left + right) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            left = mid + 1                     # too small: the target is to the right
        else:
            right = mid - 1                    # too big: the target is to the left
    return -1

assert search([-1, 0, 3, 5, 9, 12], 9) == 4
assert search([-1, 0, 3, 5, 9, 12], 2) == -1
assert search([5], 5) == 0
assert search([5], -5) == -1
assert search([1, 2], 2) == 1

rng = random.Random(0)
for _ in range(500):
    nums = sorted(rng.sample(range(-20, 20), rng.randint(1, 10)))
    t = rng.randint(-21, 21)
    assert search(nums, t) == (nums.index(t) if t in nums else -1)
print(search([-1, 0, 3, 5, 9, 12], 9))

In [ ]:
def search_movie(nums, target):
    m = Movie("Binary Search", f"Where is {target}? Guess the middle, then throw half away")
    left, right, n = 0, len(nums) - 1, len(nums)
    m.step(f"The cards are sorted. `{target}` can only be between L and R. Higher or lower — let's play!",
           arr(nums, ptr={"L": left, "R": right}), note(target=target))
    while left <= right:
        mid = (left + right) // 2
        grey, ptr = grey_outside(n, left, right), {"L": left, "M": mid, "R": right}
        if nums[mid] == target:
            m.step(f"The middle card M is `{nums[mid]}`. That's it! Found at index **{mid}** ✅",
                   arr(nums, hi={**grey, mid: "good"}, ptr=ptr), note(target=target))
            return mid, m
        m.step(f"Guess the middle card M: `{nums[mid]}`.", arr(nums, hi={**grey, mid: "look"}, ptr=ptr), note(target=target))
        if nums[mid] < target:
            m.step(f"`{nums[mid]}` < `{target}`: **higher!** M and everything left of it is too small. Throw them away: L = {mid + 1}.",
                   arr(nums, hi={**grey, **{k: "bad" for k in range(left, mid + 1)}}, ptr=ptr), note(target=target))
            left = mid + 1
        else:
            m.step(f"`{nums[mid]}` > `{target}`: **lower!** M and everything right of it is too big. Throw them away: R = {mid - 1}.",
                   arr(nums, hi={**grey, **{k: "bad" for k in range(mid, right + 1)}}, ptr=ptr), note(target=target))
            right = mid - 1
    m.step(f"L passed R: nothing is left to search, so `{target}` is not here → **-1**.",
           arr(nums, hi=grey_outside(n, 1, 0)), note(target=target))
    return -1, m

nums, target = [1, 3, 4, 6, 8, 9, 11, 14, 17], 14
answer, m = search_movie(nums, target)
assert answer == search(nums, target) == 7
m.show()

---

## 35 · Search Insert Position · [LeetCode 35](https://leetcode.com/problems/search-insert-position/) · 🟢 Easy

**🧸 Like you're five:** Your books stand on a shelf in order of height. Where do you slide in a new book so the order stays
        right? Play higher-or-lower; when the game ends without finding it, **L** is pointing at the gap.

**The question:** Given a **sorted** list of distinct integers and a `target`, return its index if found; otherwise return the index where it would be inserted to keep the list sorted. **O(log n)** time.

| Input | Output |
|---|---|
| `nums = [1, 3, 5, 6], target = 5` | `2` |
| `nums = [1, 3, 5, 6], target = 2` | `1` |
| `nums = [1, 3, 5, 6], target = 7` | `4` |

**🔎 How to spot it:** "**where would it go**", "first element ≥ target", "insert position", "lower bound" → binary
        search, and return `left` when it stops. (Python's `bisect.bisect_left` does exactly this.)

**💡 The trick (binary search):** Run the normal binary search from problem 34. If you find the target, return `mid`. If not, the loop
        ends with `left = right + 1`, and `left` is the first position whose value is bigger than the target — the
        gap. The trap: returning `mid` or `right` instead of `left`; also remember the answer can be `len(nums)` (after
        everything).

**⏱ Cost:** **O(log n)** time · **O(1)** space. Brute force: walk until you meet a value ≥ target — O(n).

In [ ]:
def search_insert(nums, target):
    left, right = 0, len(nums) - 1
    while left <= right:
        mid = (left + right) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return left                        # everything before left is smaller, everything from left on is bigger

assert search_insert([1, 3, 5, 6], 5) == 2
assert search_insert([1, 3, 5, 6], 2) == 1
assert search_insert([1, 3, 5, 6], 7) == 4     # after everything
assert search_insert([1, 3, 5, 6], 0) == 0     # before everything
assert search_insert([1], 1) == 0

import bisect
rng = random.Random(0)
for _ in range(500):
    nums = sorted(rng.sample(range(-20, 20), rng.randint(1, 10)))
    t = rng.randint(-22, 22)
    assert search_insert(nums, t) == bisect.bisect_left(nums, t)
print(search_insert([1, 3, 5, 6], 2))

In [ ]:
def search_insert_movie(nums, target):
    m = Movie("Search Insert Position", f"Where does {target} belong on the shelf?")
    left, right, n = 0, len(nums) - 1, len(nums)
    m.step(f"Play higher-or-lower for `{target}`. If it is missing, L will end on the gap where it belongs.",
           arr(nums, ptr={"L": left, "R": right}), note(target=target))
    while left <= right:
        mid = (left + right) // 2
        grey, ptr = grey_outside(n, left, right), {"L": left, "M": mid, "R": right}
        if nums[mid] == target:
            m.step(f"M is `{nums[mid]}` — found it at index **{mid}** ✅", arr(nums, hi={**grey, mid: "good"}, ptr=ptr), note(target=target))
            return mid, m
        if nums[mid] < target:
            m.step(f"M = `{nums[mid]}` is smaller than `{target}`. Throw away M and its left: L = {mid + 1}.",
                   arr(nums, hi={**grey, **{k: "bad" for k in range(left, mid + 1)}}, ptr=ptr), note(target=target))
            left = mid + 1
        else:
            m.step(f"M = `{nums[mid]}` is bigger than `{target}`. Throw away M and its right: R = {mid - 1}.",
                   arr(nums, hi={**grey, **{k: "bad" for k in range(mid, right + 1)}}, ptr=ptr), note(target=target))
            right = mid - 1
    m.step(f"Now R = {right} is left of L = {left}: nothing left. Everything before L is smaller, everything from L on is bigger.",
           arr(nums, hi={**{k: "done" for k in range(n)}, **({left: "look"} if left < n else {})}, ptr={"R": right, "L": left}),
           note(target=target, L=left, R=right))
    shelf = nums[:left] + [target] + nums[left:]
    m.step(f"So `{target}` slides in at index **{left}** ✅", arr(shelf, "the shelf after inserting", hi={left: "good"}, ptr={"here": left}))
    return left, m

nums, target = [1, 3, 5, 6, 8, 10, 12], 7
answer, m = search_insert_movie(nums, target)
assert answer == search_insert(nums, target) == 4
m.show()

---

## 36 · Search a 2D Matrix · [LeetCode 74](https://leetcode.com/problems/search-a-2d-matrix/) · 🟡 Medium

**🧸 Like you're five:** A chocolate box whose pieces are numbered in reading order: left to right, then the next row. If you
        read the rows one after another, it is just one long sorted line — so play higher-or-lower on the line.

**The question:** Each row of the `m × n` matrix is sorted, and each row's first number is bigger than the last number of the row above. Return whether `target` is in the matrix, in **O(log(m·n))** time.

| Input | Output |
|---|---|
| `matrix = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], target = 3` | `True` |
| `same matrix, target = 13` | `False` |
| `matrix = [[1]], target = 2` | `False` |

**🔎 How to spot it:** A 2-D grid that is sorted **in reading order** (row after row) → it is secretly a 1-D sorted list.
        (If only each row and each column is sorted — LeetCode 240 — start at the top-right corner instead.)

**💡 The trick (binary search):** Binary search over positions `0 … m·n − 1`. Turn a position into a cell with `row, col =
        divmod(mid, n)` — divide by the number of **columns**, not rows (the usual slip). Everything else is problem
        34.

**⏱ Cost:** **O(log(m·n))** time · **O(1)** space. Searching every cell is O(m·n); binary-searching the row, then the column, is also O(log m + log n).

In [ ]:
def search_matrix(matrix, target):
    rows, cols = len(matrix), len(matrix[0])
    left, right = 0, rows * cols - 1           # pretend the grid is one long sorted row
    while left <= right:
        mid = (left + right) // 2
        r, c = divmod(mid, cols)               # position -> (row, column): divide by the number of COLUMNS
        x = matrix[r][c]
        if x == target:
            return True
        if x < target:
            left = mid + 1
        else:
            right = mid - 1
    return False

grid_ = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]]
assert search_matrix(grid_, 3) is True
assert search_matrix(grid_, 13) is False
assert search_matrix(grid_, 60) is True and search_matrix(grid_, 1) is True   # both corners
assert search_matrix([[1]], 2) is False
assert search_matrix([[1, 3]], 3) is True       # one row
assert search_matrix([[1], [3]], 3) is True     # one column

rng = random.Random(0)
for _ in range(300):
    r, c = rng.randint(1, 4), rng.randint(1, 4)
    flat = sorted(rng.sample(range(50), r * c))
    mat = [flat[i * c:(i + 1) * c] for i in range(r)]
    t = rng.randint(0, 50)
    assert search_matrix(mat, t) == (t in flat)
print(search_matrix(grid_, 3))

In [ ]:
def search_matrix_movie(matrix, target):
    m = Movie("Search a 2D Matrix", f"Is {target} in the box? Read it as one long row")
    rows, cols = len(matrix), len(matrix[0])
    flat = [x for row in matrix for x in row]
    left, right = 0, rows * cols - 1
    def pics(hi, ptr):
        cells = {divmod(k, cols): role for k, role in hi.items()}
        return (grid(matrix, "the box", hi=cells, row_labels=range(rows), col_labels=range(cols)),
                arr(flat, "the same numbers as one long row", hi=hi, ptr=ptr))
    m.step(f"Read the rows one after another: it is one sorted row of {len(flat)}. Binary search it for `{target}`.",
           *pics({}, {"L": left, "R": right}), note(target=target))
    while left <= right:
        mid = (left + right) // 2
        r, c = divmod(mid, cols)
        grey, ptr = grey_outside(len(flat), left, right), {"L": left, "M": mid, "R": right}
        where = f"Position {mid} is row {mid} // {cols} = {r}, column {mid} % {cols} = {c}"
        if flat[mid] == target:
            m.step(f"{where}: `{flat[mid]}`. Found it → **True** ✅", *pics({**grey, mid: "good"}, ptr), note(target=target))
            return True, m
        if flat[mid] < target:
            m.step(f"{where}: `{flat[mid]}`, too small. Throw it away, and everything before it.",
                   *pics({**grey, **{k: "bad" for k in range(left, mid + 1)}}, ptr), note(target=target))
            left = mid + 1
        else:
            m.step(f"{where}: `{flat[mid]}`, too big. Throw it away, and everything after it.",
                   *pics({**grey, **{k: "bad" for k in range(mid, right + 1)}}, ptr), note(target=target))
            right = mid - 1
    m.step(f"L passed R: `{target}` is not in the box → **False**.", *pics(grey_outside(len(flat), 1, 0), {}), note(target=target))
    return False, m

matrix, target = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 30
answer, m = search_matrix_movie(matrix, target)
assert answer == search_matrix(matrix, target) is True
m.show()

---

## 37 · Find Minimum in Rotated Sorted Array · [LeetCode 153](https://leetcode.com/problems/find-minimum-in-rotated-sorted-array/) · 🟡 Medium

**🧸 Like you're five:** Stairs going up were cut in two and the pieces swapped, so now they climb, fall off a **cliff**, and climb
        again. The lowest step is at the bottom of the cliff. Look at the middle step and the last step to know which
        side the cliff is on.

**The question:** A sorted list of **distinct** numbers was rotated (e.g. `[0,1,2,4,5,6,7]` → `[4,5,6,7,0,1,2]`). Return the minimum in **O(log n)** time.

| Input | Output |
|---|---|
| `nums = [3, 4, 5, 1, 2]` | `1` |
| `nums = [4, 5, 6, 7, 0, 1, 2]` | `0` |
| `nums = [11, 13, 15, 17]` | `11` |

**🔎 How to spot it:** "**sorted, then rotated**" + "O(log n)" → binary search, comparing `nums[mid]` with an **end** of the
        range to learn which half is the plain sorted one.

**💡 The trick (binary search):** Compare `nums[mid]` with `nums[right]`. If `nums[mid] > nums[right]`, the numbers drop somewhere after
        `mid`, so the minimum is to the right: `left = mid + 1`. Otherwise `mid..right` climbs nicely, so the minimum is
        `mid` **or** to its left: `right = mid` (not `mid − 1` — `mid` might be the answer). Stop when `left ==
        right`. The trap: comparing with `nums[left]` instead — that fails when the list is not rotated at all.

**⏱ Cost:** **O(log n)** time · **O(1)** space. Brute force: `min(nums)` — O(n).

In [ ]:
def find_min(nums):
    left, right = 0, len(nums) - 1
    while left < right:                    # stop when one number is left
        mid = (left + right) // 2
        if nums[mid] > nums[right]:        # the cliff (the drop) is to the right of mid
            left = mid + 1
        else:                              # mid..right climbs: the minimum is mid or left of it
            right = mid
    return nums[left]

assert find_min([3, 4, 5, 1, 2]) == 1
assert find_min([4, 5, 6, 7, 0, 1, 2]) == 0
assert find_min([11, 13, 15, 17]) == 11     # not rotated at all
assert find_min([1]) == 1
assert find_min([2, 1]) == 1

rng = random.Random(0)
for _ in range(500):
    base = sorted(rng.sample(range(-30, 30), rng.randint(1, 10)))
    k = rng.randrange(len(base))
    nums = base[k:] + base[:k]
    assert find_min(nums) == min(nums)
print(find_min([4, 5, 6, 7, 0, 1, 2]))

In [ ]:
def find_min_movie(nums):
    m = Movie("Find Minimum in Rotated Sorted Array", "Find the bottom of the cliff")
    left, right, n = 0, len(nums) - 1, len(nums)
    m.step("A sorted row was turned around, so it has one **cliff** where it drops. The smallest is at the bottom of the cliff.",
           bars(nums, ptr={"L": left, "R": right}))
    while left < right:
        mid = (left + right) // 2
        grey, ptr = grey_outside(n, left, right), {"L": left, "M": mid, "R": right}
        m.step(f"Compare the middle M = `{nums[mid]}` with the right end R = `{nums[right]}`.",
               bars(nums, hi={**grey, mid: "look", right: "look"}, ptr=ptr))
        if nums[mid] > nums[right]:
            m.step(f"`{nums[mid]}` > `{nums[right]}`: the row must drop somewhere after M. The cliff is to the right, so throw away L..M.",
                   bars(nums, hi={**grey, **{k: "bad" for k in range(left, mid + 1)}}, ptr=ptr))
            left = mid + 1
        else:
            m.step(f"`{nums[mid]}` ≤ `{nums[right]}`: M..R only climbs, so the bottom is M or to its left. Keep M, throw away the rest.",
                   bars(nums, hi={**grey, **{k: "bad" for k in range(mid + 1, right + 1)}}, ptr=ptr))
            right = mid
    m.step(f"L and R meet at index {left}: the smallest number is **{nums[left]}** ✅",
           bars(nums, hi={**grey_outside(n, left, left), left: "good"}, ptr={"L": left, "R": right}))
    return nums[left], m

nums = [5, 6, 7, 8, 9, 1, 2, 3, 4]
answer, m = find_min_movie(nums)
assert answer == find_min(nums) == 1
m.show()

---

## 38 · Search in Rotated Sorted Array · [LeetCode 33](https://leetcode.com/problems/search-in-rotated-sorted-array/) · 🟡 Medium

**🧸 Like you're five:** The same cut-and-swapped stairs, but now you hunt for one step. When you look at the middle, at least
        one side of it is a **tidy** climb with no cliff. It is easy to tell whether your number lives on the tidy
        side; if not, it must be on the other side.

**The question:** A sorted list of **distinct** integers was rotated at an unknown point. Return the index of `target`, or `-1`, in **O(log n)** time.

| Input | Output |
|---|---|
| `nums = [4, 5, 6, 7, 0, 1, 2], target = 0` | `4` |
| `nums = [4, 5, 6, 7, 0, 1, 2], target = 3` | `-1` |
| `nums = [1], target = 0` | `-1` |

**🔎 How to spot it:** "rotated sorted" + "search" + "O(log n)" → binary search where each step first asks **which half is
        sorted**.

**💡 The trick (binary search):** At each `mid`: if `nums[left] <= nums[mid]`, the **left** half is sorted — the target is there exactly
        when `nums[left] <= target < nums[mid]`. Otherwise the **right** half is sorted — the target is there exactly
        when `nums[mid] < target <= nums[right]`. Keep the half that can hold it. The trap: using `<` instead of `<=`
        in `nums[left] <= nums[mid]` breaks when `left == mid` (a two-item range).

**⏱ Cost:** **O(log n)** time · **O(1)** space. Brute force: scan every item — O(n). (Finding the cliff first with problem 37, then a normal search, is also O(log n).)

In [ ]:
def search_rotated(nums, target):           # LeetCode calls it search
    left, right = 0, len(nums) - 1
    while left <= right:
        mid = (left + right) // 2
        if nums[mid] == target:
            return mid
        if nums[left] <= nums[mid]:                 # the LEFT half left..mid is sorted
            if nums[left] <= target < nums[mid]:
                right = mid - 1                     # the target is inside the sorted left half
            else:
                left = mid + 1
        else:                                       # the RIGHT half mid..right is sorted
            if nums[mid] < target <= nums[right]:
                left = mid + 1                      # the target is inside the sorted right half
            else:
                right = mid - 1
    return -1

assert search_rotated([4, 5, 6, 7, 0, 1, 2], 0) == 4
assert search_rotated([4, 5, 6, 7, 0, 1, 2], 3) == -1
assert search_rotated([1], 0) == -1
assert search_rotated([3, 1], 1) == 1        # the two-item case that needs <=
assert search_rotated([5, 1, 3], 5) == 0

rng = random.Random(0)
for _ in range(500):
    base = sorted(rng.sample(range(-20, 20), rng.randint(1, 10)))
    k = rng.randrange(len(base))
    nums, t = base[k:] + base[:k], rng.randint(-21, 21)
    assert search_rotated(nums, t) == (nums.index(t) if t in nums else -1)
print(search_rotated([4, 5, 6, 7, 0, 1, 2], 0))

In [ ]:
def search_rotated_movie(nums, target):
    m = Movie("Search in Rotated Sorted Array", f"Find {target}: one half is always a tidy climb")
    left, right, n = 0, len(nums) - 1, len(nums)
    m.step(f"The row climbs, falls off a cliff, climbs again. Find `{target}` between L and R.",
           arr(nums, ptr={"L": left, "R": right}), note(target=target))
    while left <= right:
        mid = (left + right) // 2
        grey, ptr = grey_outside(n, left, right), {"L": left, "M": mid, "R": right}
        if nums[mid] == target:
            m.step(f"M is `{nums[mid]}` — found it at index **{mid}** ✅", arr(nums, hi={**grey, mid: "good"}, ptr=ptr), note(target=target))
            return mid, m
        if nums[left] <= nums[mid]:
            side, lo, hi_ = "left", left, mid
            inside = nums[left] <= target < nums[mid]
        else:
            side, lo, hi_ = "right", mid, right
            inside = nums[mid] < target <= nums[right]
        m.step(f"M = `{nums[mid]}`, not {target}. The **{side}** half `{nums[lo]} … {nums[hi_]}` is a tidy climb (blue).",
               arr(nums, hi={**grey, **{k: "win" for k in range(lo, hi_ + 1)}, mid: "look"}, ptr=ptr), note(target=target, tidy=side))
        keep_left = (side == "left") == inside
        throw = range(mid, right + 1) if keep_left else range(left, mid + 1)
        verdict = "Yes" if inside else "No"
        m.step(f"Is `{target}` inside `{nums[lo]} … {nums[hi_]}`? {verdict}. So throw away the **{'right' if keep_left else 'left'}** side.",
               arr(nums, hi={**grey, **{k: "bad" for k in throw}}, ptr=ptr), note(target=target, tidy=side))
        if keep_left:
            right = mid - 1
        else:
            left = mid + 1
    m.step(f"L passed R: `{target}` is not here → **-1**.", arr(nums, hi=grey_outside(n, 1, 0)), note(target=target))
    return -1, m

nums, target = [6, 7, 8, 9, 1, 2, 3, 4, 5], 2
answer, m = search_rotated_movie(nums, target)
assert answer == search_rotated(nums, target) == 5
m.show()

---

## 39 · Find First and Last Position of Element in Sorted Array · [LeetCode 34](https://leetcode.com/problems/find-first-and-last-position-of-element-in-sorted-array/) · 🟡 Medium

**🧸 Like you're five:** A row of sorted cards has several 8s in a row. Find where the 8s **start** and where they **end**. When
        you hit an 8, don't stop — write it down and keep playing higher-or-lower towards the edge you are hunting.

**The question:** Given a sorted list `nums` (duplicates allowed) and a `target`, return `[first index, last index]` of `target`, or `[-1, -1]` if it is missing. **O(log n)** time.

| Input | Output |
|---|---|
| `nums = [5, 7, 7, 8, 8, 10], target = 8` | `[3, 4]` |
| `nums = [5, 7, 7, 8, 8, 10], target = 6` | `[-1, -1]` |
| `nums = [], target = 0` | `[-1, -1]` |

**🔎 How to spot it:** "**first / last** occurrence", "how many times does x appear" in a **sorted** list → two binary
        searches that do not stop at the first hit.

**💡 The trick (binary search):** Write one binary search with a switch. On a hit, remember `mid`, then keep going: towards the left
        (`right = mid − 1`) when hunting the **first** one, towards the right (`left = mid + 1`) when hunting the
        **last**. Run it twice. The trap: finding one 8 and then walking left and right one step at a time — that is
        O(n) when every card is an 8.

**⏱ Cost:** **O(log n)** time (two binary searches) · **O(1)** space. Brute force: scan the whole list — O(n).

In [ ]:
def search_range(nums, target):
    def find(go_left):
        left, right, found = 0, len(nums) - 1, -1
        while left <= right:
            mid = (left + right) // 2
            if nums[mid] < target:
                left = mid + 1
            elif nums[mid] > target:
                right = mid - 1
            else:
                found = mid                   # a hit! write it down ...
                if go_left:
                    right = mid - 1           # ... and keep looking further left for the first one
                else:
                    left = mid + 1            # ... or further right for the last one
        return found
    return [find(True), find(False)]

assert search_range([5, 7, 7, 8, 8, 10], 8) == [3, 4]
assert search_range([5, 7, 7, 8, 8, 10], 6) == [-1, -1]
assert search_range([], 0) == [-1, -1]
assert search_range([1], 1) == [0, 0]
assert search_range([2, 2, 2, 2], 2) == [0, 3]     # every card is the target

rng = random.Random(0)
for _ in range(500):
    nums = sorted(rng.randint(0, 5) for _ in range(rng.randint(0, 10)))
    t = rng.randint(-1, 6)
    spots = [i for i, x in enumerate(nums) if x == t]
    assert search_range(nums, t) == ([spots[0], spots[-1]] if spots else [-1, -1])
print(search_range([5, 7, 7, 8, 8, 10], 8))

In [ ]:
def search_range_movie(nums, target):
    m = Movie("Find First and Last Position", f"Where do the {target}s start and end?")
    n, ans = len(nums), []
    m.step(f"Two games of higher-or-lower: one hunts the **first** `{target}`, one hunts the **last**.", arr(nums), note(target=target))
    for go_left in (True, False):
        word, left, right, found = ("first" if go_left else "last"), 0, n - 1, -1
        while left <= right:
            mid = (left + right) // 2
            grey, ptr = grey_outside(n, left, right), {"L": left, "M": mid, "R": right}
            if nums[mid] < target:
                throw, cap = range(left, mid + 1), f"Hunting the {word}: M = `{nums[mid]}` is too small. Throw away M and its left."
                left = mid + 1
            elif nums[mid] > target:
                throw, cap = range(mid, right + 1), f"Hunting the {word}: M = `{nums[mid]}` is too big. Throw away M and its right."
                right = mid - 1
            else:
                found = mid
                side = "left" if go_left else "right"
                cap = f"M = `{target}`, a hit at {mid}! Write it down, but the **{word}** one may be further {side}. Keep going {side}."
                throw = range(mid + 1, right + 1) if go_left else range(left, mid)
                if go_left:
                    right = mid - 1
                else:
                    left = mid + 1
            hi = {**grey, **{k: "bad" for k in throw}, **({found: "good"} if found != -1 else {})}
            m.step(cap, arr(nums, hi=hi, ptr=ptr), note(target=target, **{word: found}))
        ans.append(found)
        m.step(f"L passed R, so the hunt is over: the **{word}** `{target}` is at index **{found}**.",
               arr(nums, hi={found: "good"} if found != -1 else {}), note(target=target, **{word: found}))
    a, b = ans
    m.step(f"Answer: **{ans}** — the `{target}`s run from index {a} to index {b} ✅",
           arr(nums, hi={k: "good" for k in range(a, b + 1)}, ptr={"first": a, "last": b}), note(target=target))
    return ans, m

nums, target = [5, 7, 7, 8, 8, 8, 8, 10], 8
answer, m = search_range_movie(nums, target)
assert answer == search_range(nums, target) == [3, 6]
m.show()

---

## 40 · Koko Eating Bananas · [LeetCode 875](https://leetcode.com/problems/koko-eating-bananas/) · 🟡 Medium

**🧸 Like you're five:** Koko the monkey has piles of bananas and the guard is back in `h` hours. Each hour she eats up to `k`
        bananas from **one** pile. She likes to eat slowly. Guess a speed: too slow? Try faster. Fast enough? Try
        slower. It is higher-or-lower again — but on the **speed**, not on the piles.

**The question:** Given `piles` and `h ≥ len(piles)`, return the smallest integer speed `k` (bananas per hour) such that Koko eats every banana within `h` hours. In one hour she eats `k` bananas from one pile, or the whole pile if it has fewer.

| Input | Output |
|---|---|
| `piles = [3, 6, 7, 11], h = 8` | `4` |
| `piles = [30, 11, 23, 4, 20], h = 5` | `30` |
| `piles = [30, 11, 23, 4, 20], h = 6` | `23` |

**🔎 How to spot it:** "the **minimum** speed / capacity / size such that it **still works**", and a faster speed never hurts
        → **binary search on the answer**. You are not searching the list; you are searching the range of possible
        answers, `1 … max(piles)`.

**💡 The trick (binary search on the answer):** 1. The answer is between `1` and `max(piles)` (at that speed every pile takes one hour).
        2. `hours(k) = sum(ceil(p / k))` — a pile of 7 at speed 4 takes **2** hours, not 1.75.
        3. Binary search: if `hours(mid) <= h`, `mid` works — keep it and try slower (`right = mid`); otherwise it is
        too slow (`left = mid + 1`). When `left == right`, that is the slowest speed that works.
        The trap: using `p // k` instead of rounding **up**, or starting at `0` (divide by zero).

**⏱ Cost:** **O(n · log M)** time, where M = `max(piles)` (log M guesses, each adds up n piles) · **O(1)** space. Brute force: try every speed from 1 up — O(n · M).

In [ ]:
def min_eating_speed(piles, h):
    def hours(speed):
        return sum((p + speed - 1) // speed for p in piles)   # round UP: a half-eaten pile still costs a whole hour
    left, right = 1, max(piles)          # speed max(piles) always works: one pile per hour
    while left < right:
        mid = (left + right) // 2
        if hours(mid) <= h:
            right = mid                  # fast enough: keep it, and try slower
        else:
            left = mid + 1               # too slow: must eat faster
    return left

assert min_eating_speed([3, 6, 7, 11], 8) == 4
assert min_eating_speed([30, 11, 23, 4, 20], 5) == 30
assert min_eating_speed([30, 11, 23, 4, 20], 6) == 23
assert min_eating_speed([1], 1) == 1
assert min_eating_speed([1000000000], 2) == 500000000       # huge pile: still ~30 guesses

rng = random.Random(0)
for _ in range(300):
    piles = [rng.randint(1, 30) for _ in range(rng.randint(1, 6))]
    h = rng.randint(len(piles), 40)
    brute = next(k for k in range(1, 31) if sum(-(-p // k) for p in piles) <= h)
    assert min_eating_speed(piles, h) == brute
print(min_eating_speed([3, 6, 7, 11], 8))

In [ ]:
def min_eating_speed_movie(piles, h):
    m = Movie("Koko Eating Bananas", f"The slowest speed that finishes every pile within {h} hours")
    speeds = list(range(1, max(piles) + 1))
    hours_at = lambda k: [(p + k - 1) // k for p in piles]
    left, right = 1, max(piles)
    pos = lambda ptr: {name: k - 1 for name, k in ptr.items()}          # a speed k sits in box k - 1
    grey = lambda: {k - 1: "done" for k in speeds if k < left or k > right}
    m.step(f"Koko can eat 1 to {max(piles)} bananas an hour. Guess a speed: fast enough? Try slower. Too slow? Try faster.",
           arr(speeds, "speeds (bananas per hour)", ptr=pos({"L": left, "R": right}), index=False), arr(piles, "banana piles", index=False), note(h=h))
    while left < right:
        mid = (left + right) // 2
        hrs = hours_at(mid)
        ok = sum(hrs) <= h
        ptr = pos({"L": left, "M": mid, "R": right})
        m.step(f"Try speed **{mid}**: the piles take {' + '.join(map(str, hrs))} = `{sum(hrs)}` hours. " +
               (f"That fits in {h}!" if ok else f"More than {h}: too slow."),
               arr(speeds, "speeds (bananas per hour)", hi={**grey(), mid - 1: "good" if ok else "bad"}, ptr=ptr, index=False),
               arr(piles, "banana piles", index=False), arr(hrs, f"hours per pile at speed {mid}", hi={i: "look" for i in range(len(hrs))}, index=False),
               note(hours=sum(hrs), h=h))
        if ok:
            m.step(f"Speed {mid} works, so every faster speed works too. Keep {mid}, throw away the faster ones: R = {mid}.",
                   arr(speeds, "speeds (bananas per hour)", hi={**grey(), mid - 1: "good", **{k - 1: "bad" for k in range(mid + 1, right + 1)}}, ptr=ptr, index=False),
                   arr(piles, "banana piles", index=False), note(h=h))
            right = mid
        else:
            m.step(f"Speed {mid} is too slow, so every slower speed is too slow too. Throw them away: L = {mid + 1}.",
                   arr(speeds, "speeds (bananas per hour)", hi={**grey(), **{k - 1: "bad" for k in range(left, mid + 1)}}, ptr=ptr, index=False),
                   arr(piles, "banana piles", index=False), note(h=h))
            left = mid + 1
    hrs = hours_at(left)
    m.step(f"L and R meet: the slowest speed that works is **{left}** bananas an hour ({sum(hrs)} hours) ✅",
           arr(speeds, "speeds (bananas per hour)", hi={**grey(), left - 1: "good"}, ptr=pos({"L": left, "R": right}), index=False),
           arr(piles, "banana piles", index=False), arr(hrs, f"hours per pile at speed {left}", hi={i: "good" for i in range(len(hrs))}, index=False),
           note(hours=sum(hrs), h=h))
    return left, m

piles, h = [3, 6, 7, 11], 8
answer, m = min_eating_speed_movie(piles, h)
assert answer == min_eating_speed(piles, h) == 4
m.show()

## Quiz yourself: which trick?

Eight questions that are **not** in the top 100. Read one line, decide the trick, then open the answer.

1. [643 · Maximum Average Subarray I](https://leetcode.com/problems/maximum-average-subarray-i/) (🟢 Easy) — the highest average of any `k` numbers that sit next to each other.
   <details><summary>which trick?</summary>A <b>fixed-size sliding window</b>: add the number that walks in, subtract the one that walks out — O(n) instead of re-adding k numbers each time.</details>
2. [1004 · Max Consecutive Ones III](https://leetcode.com/problems/max-consecutive-ones-iii/) (🟡 Medium) — the longest run of 1s if you may flip at most `k` zeros.
   <details><summary>which trick?</summary>A <b>variable sliding window</b> that allows at most k zeros inside; shrink from the left while there are too many. It is problem 24 with only two letters.</details>
3. [1047 · Remove All Adjacent Duplicates In String](https://leetcode.com/problems/remove-all-adjacent-duplicates-in-string/) (🟢 Easy) — keep deleting two equal letters that stand side by side.
   <details><summary>which trick?</summary>A <b>stack</b>: if the new letter equals the top, pop it (the pair vanishes); otherwise push. Deleting a pair can make a new pair meet — the stack handles that for free.</details>
4. [496 · Next Greater Element I](https://leetcode.com/problems/next-greater-element-i/) (🟢 Easy) — for each number, the first bigger number to its right.
   <details><summary>which trick?</summary>A <b>monotonic stack</b>, exactly like Daily Temperatures (31): waiting numbers get their answer when a bigger one arrives. Store the answers in a dictionary.</details>
5. [71 · Simplify Path](https://leetcode.com/problems/simplify-path/) (🟡 Medium) — turn `/a/./b/../../c/` into `/c`.
   <details><summary>which trick?</summary>A <b>stack</b> of folder names: a name is pushed, <code>..</code> pops, <code>.</code> and empty parts are skipped. Join the stack with <code>/</code> at the end.</details>
6. [744 · Find Smallest Letter Greater Than Target](https://leetcode.com/problems/find-smallest-letter-greater-than-target/) (🟢 Easy) — in sorted letters, the first letter bigger than `target` (wrapping around).
   <details><summary>which trick?</summary><b>Binary search</b> for the first position with a letter &gt; target (like Search Insert, 35); if it falls off the end, wrap to index 0.</details>
7. [1011 · Capacity To Ship Packages Within D Days](https://leetcode.com/problems/capacity-to-ship-packages-within-d-days/) (🟡 Medium) — the smallest ship that carries every package, in order, within `days` days.
   <details><summary>which trick?</summary><b>Binary search on the answer</b>, just like Koko (40): capacities from max(weights) to sum(weights); a bigger ship never needs more days.</details>
8. [69 · Sqrt(x)](https://leetcode.com/problems/sqrtx/) (🟢 Easy) — the whole-number square root of `x`, without `**` or `math.sqrt`.
   <details><summary>which trick?</summary><b>Binary search on the answer</b>: find the largest k with k·k ≤ x. The answers 0 … x are sorted by "k·k is too big or not".</details>

## Pattern cheat sheet

| If the question says… | think… | example here |
|---|---|---|
| "buy before you sell", "biggest later − earlier" | remember the smallest so far | 21 |
| "as many trades as you like" | add up every rise | 22 |
| "longest / shortest **substring** or **subarray** such that…" | variable sliding window: grow right, shrink left | 23, 24, 26 |
| "at most **k** changes / flips" | window rule `size − most common ≤ k` | 24 |
| "a **permutation / anagram** of p inside s" | fixed-size window + letter counts | 25, 27 |
| "matching / nested brackets", "decode k[…]" | stack of things still open | 28, 32 |
| "evaluate postfix / an expression" | stack of numbers | 30 |
| "**next greater / warmer** element", "how many days until…" | monotonic stack of indices | 31 |
| "design a stack / queue that also does X in O(1)" | a second stack alongside | 29, 33 |
| "**sorted** … find … O(log n)" | binary search | 34 |
| "where would it be inserted", "first ≥ target" | binary search, return `left` | 35 |
| "sorted 2-D matrix, row after row" | one long row: `divmod(mid, cols)` | 36 |
| "sorted but **rotated**" | binary search; one half is always sorted | 37, 38 |
| "**first and last** position", "count of x" | binary search that keeps going after a hit | 39 |
| "**minimum speed / capacity / days** such that it works" | binary search on the answer | 40 |

## Say it in an interview

- **Clarify first.** "Are the numbers all positive?" (the sliding window in 26 needs it) · "Is the list sorted?
  Distinct?" (binary search needs sorted; 37 and 38 need distinct) · "Can the input be empty?"
- **Say the brute force, then name the trick.** "Checking every substring is O(n²). The window only moves
  forward, so each letter enters and leaves once — O(n)." Or: "The list is sorted, so I can throw half away
  each step — O(log n)."
- **Windows:** say out loud *what the window promises* ("no repeats inside", "at most k to repaint") and when you
  shrink it. Use `while`, not `if`, for the shrink. Delete zero counts before comparing counters.
- **Stacks:** say *what is waiting on the stack* ("openers not yet closed", "days that have not met a warmer day").
  Push indices when you need distances. Check the stack is empty at the end.
- **Binary search:** say the *invariant* — "the answer is always inside `left..right`" — and pick one template:
  `while left <= right` with `mid ± 1`, or `while left < right` with `right = mid`. Mixing them loops forever.
- **Test the edges aloud:** one element, target smaller than everything, target bigger than everything, all
  items equal, a list that is not rotated at all.

## Next

- ⬅️ Previous: [15 · Top 100, part 1 — arrays, hashing and two pointers](15_top100_part1_arrays_hashing_two_pointers.ipynb)
- ➡️ Next: [17 · Top 100, part 3 — linked lists and trees](17_top100_part3_linked_lists_and_trees.ipynb)
- Go deeper on the patterns: [03 · Hashing and sliding window](03_hashing_and_sliding_window.ipynb) ·
  [04 · Stacks and queues](04_stacks_and_queues.ipynb) · [06 · Binary search](06_binary_search.ipynb)